<a href="https://colab.research.google.com/github/eeeewyz/LLM/blob/main/7_grpo_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graded Lab: GRPO Post Training Lab

Welcome to the second assignment of this module!

Carefully read each Markdown (text) cell, which include instructions and hints. Start by reading the background behind your upcoming tasks.

When you are done, submit your solution by saving it, then clicking on the submit button at the top right side of the page.

## In order for your submission to be graded correctly, you **MUST**:
* **Use the provided variable names**, otherwise the autograder will not be able to locate the variable for grading.

* **Replace any instances of `None` with your own code.**

* **Only modify the cells that start with the comment `# GRADED CELL`**.  

* **Use the provided cells for your solution.** You can add new cells to experiment, but these will be omitted when grading.

To submit your solution, save it, then click on the blue submit button at the top of the page.

<div style="background-color: #FAD888; padding: 10px; border-radius: 3px; box-shadow: 0 2px 4px rgba(0, 0, 0, 0.1); width:95%
">
<strong>Important notes</strong>:

- Code blocks with None will not run properly. If you run them before completing the exercise, you will likely get an error.

- The notebooks work best in Chrome browser. If you are having problems, please switch to Chrome.

- Make sure you always save before submitting.
</div>

## Introduction

In this hands-on tutorial, you'll learn how to improve a Large Language Model's ability to solve math problems using a technique called **GRPO**. This involves generating multiple answers to the same question, comparing answers to find which are better, and teaching the model to prefer better approaches.

## Objectives

You will build a comprehensive reward system for training a Large Language Model to solve math problems using GRPO (Group Relative Policy Optimization). This involves creating sophisticated reward functions that can evaluate model responses across different quality levels and implementing a complete GRPO training pipeline.

* **Extract Numerical Answers from Model Responses:** Implement robust parsing to extract numerical answers from various response formats including GSM8K standard format and common answer phrases.
* **Analyze Response Quality Indicators:** Build a quality analysis system that detects mathematical reasoning, step-by-step thinking, and structured solutions in model responses.
* **Reward Unparseable Responses:** Create a reward system for responses without clear numerical answers, based on effort and reasoning quality.
* **Reward High-Quality Correct Answers:** Implement a bonus system that encourages not just correctness but also clear mathematical communication and detailed explanations.
* **Implement Partial Credit for Wrong Answers:** Develop a partial credit system that provides learning gradients for wrong answers based on proximity to correct solutions and quality of reasoning shown.

为数学题 LLM 构建一套用于 GRPO 训练的 Reward System，让模型不仅追求“答案正确”，还会因为“推理过程好、表达清楚、接近正确答案”获得不同程度的奖励。


模型回答
   ↓
提取最终答案
   ↓
分析推理质量
   ↓
判断答案是否正确
   ↓
正确 → 基础奖励 + 高质量 bonus
错误 → 根据接近程度 + 推理质量给部分奖励
无法解析 → 根据推理努力程度给少量奖励
   ↓
得到 Reward
   ↓
用于 GRPO 更新模型

## Table of Contents

* [Setup](#setup)
* [Training Configuration](#trainingconfiguration)
* [Load the GSM8K dataset](#loadGSM8K)
* [Create the Reward Function](#createrewardfunction) - Exercise 1, 2, 3, 4, 5
* [Load the Language Model](#loadthelanguagemodel)
* [Prepare Training and Validation datasets](#preparetraining)
* [Create Evaluation Callback](#createevaluation)
* [Configure GRPO Trainer](#configuregrpotrainer)
* [Train the Model with GRPO! (Ungraded Part)](#trainmodelwithgrpo)
* [Summary](#summary)

## Setup <a id="setup"></a>

Start by importing all the necessary packages, setting up random seeds for reproducibility, setting up the devices and the logger.

准备 GRPO 数学训练实验的运行环境：导入库、设置随机种子、选择 GPU/CPU、配置日志。
import os

# 关闭 Hugging Face datasets 的进度条
# 主要是为了避免 Jupyter 中出现进度条显示/context 错误
os.environ['HF_DATASETS_DISABLE_PROGRESS_BAR'] = '1'


# =========================
# 1. 导入基础 Python 工具
# =========================

import re          # 正则表达式：后面可以用来从模型回答中提取数字答案
import random      # Python 随机数
import logging     # 日志记录
import warnings    # 控制 warning 信息

from typing import List, Dict, Optional, Tuple  
# 类型提示，让函数输入输出更清楚

from dataclasses import dataclass, field  
# 用于定义结构化配置类


# =========================
# 2. 数值计算 / 深度学习库
# =========================

import numpy as np  
# NumPy：数值计算

import torch
# PyTorch：模型训练、tensor 运算、GPU 支持


# =========================
# 3. GRPO 训练工具
# =========================

from trl import (
    GRPOConfig,    # GRPO 的训练参数配置
    GRPOTrainer    # GRPO 的训练器，真正执行训练
)


# =========================
# 4. Hugging Face 模型工具
# =========================

from transformers import (
    AutoTokenizer,          # 加载模型对应的 tokenizer
    AutoModelForCausalLM,   # 加载生成式语言模型（LLM）
)


# =========================
# 5. 数据集工具
# =========================

from datasets import (
    load_dataset,      # 从 Hugging Face 加载数据集
    load_from_disk     # 从本地磁盘加载已经保存的数据集
)


# =========================
# 6. 项目自定义工具
# =========================

from utils import (
    setup_logging,  
    # 初始化日志系统

    load_and_explore_gsm8k_dataset,
    # 加载并查看 GSM8K 数学数据集

    prepare_dataset,
    # 对 GSM8K 数据进行预处理，准备给 GRPO 使用

    GSM8KEvaluationCallback,
    # 训练过程中自动评估 GSM8K 表现

    evaluate_and_compare              
    # 比较模型训练前后的效果
)


# =========================
# 7. 关闭 warning
# =========================

warnings.filterwarnings('ignore')
# 不显示 warning，让 notebook 输出更干净


# =========================
# 8. 固定随机种子
# =========================

random.seed(42)    
# 固定 Python 随机数

np.random.seed(42)
# 固定 NumPy 随机数

torch.manual_seed(42)
# 固定 PyTorch 随机数

torch.use_deterministic_algorithms(True)
# 尽量使用确定性算法
# 目的是让多次实验的结果尽可能一致、方便复现


# =========================
# 9. 自动选择 GPU / CPU
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 如果有 NVIDIA GPU：
# device = cuda
#
# 如果没有 GPU：
# device = cpu


# =========================
# 10. 初始化日志
# =========================

logger, log_file = setup_logging(device=device)

# logger：
# 用于后面记录训练信息
#
# log_file：
# 日志保存的文件位置


print("✅ libraries imported")
# 表示环境初始化完成

In [ ]:
import os
# Disable progress bars to avoid Jupyter context errors
os.environ['HF_DATASETS_DISABLE_PROGRESS_BAR'] = '1'
import re
import random
import logging
import warnings
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import torch
from trl import (
    GRPOConfig,
    GRPOTrainer
)
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from datasets import load_dataset, load_from_disk


from utils import (
    setup_logging,
    load_and_explore_gsm8k_dataset,
    prepare_dataset,
    GSM8KEvaluationCallback,
    evaluate_and_compare
)


warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger, log_file = setup_logging(device=device)

print("✅ libraries imported")

✅ Logging configured - all detailed logs will be written to: ./grpo_logs/grpo_20260822_035526.log
Note: Detailed reward computation logs will only appear in the log file, not in console output
✅ libraries imported


## Training Configuration <a id="trainingconfiguration"></a>

You'll configure all your training settings in one place. This makes it easy to experiment with different values.

Important Trade-offs:

- **Higher batch size** = More stable but needs more memory
- **More generations** = Better comparison but slower training
- **Higher learning rate** = Faster learning but might "overshoot"
- **More epochs** = More learning but might overfit

# ============================================
# Step 1: Define GRPO Training Configuration
# 作用：
# 把模型、训练、GRPO、数据、日志等超参数统一放在一个配置类中
# 后面训练时直接读取这些参数
# ============================================

@dataclass
class TrainingConfig:
    """
    GRPO training configuration.
    相当于整个训练过程的“控制面板”。
    """

    # ============================================
    # 1. MODEL SETTINGS
    # 模型相关设置
    # ============================================

    model_name: str = field(
        default="/app/models/deepseek-math-7b-base",
        # 预训练模型路径
        # GRPO 会从这个模型开始继续训练
        metadata={"help": "The pre-trained model to start with."}
    )

    output_dir: str = field(
        default="./grpo_finetuned_model",
        # 训练后的模型 / checkpoint 保存位置
        metadata={"help": "Where to save the trained model."}
    )


    # ============================================
    # 2. TRAINING DURATION
    # 控制训练多久
    # ============================================

    num_train_epochs: int = field(
        default=5,
        # 整个训练数据完整训练 5 遍
        # 1 epoch = 看完整个 training set 一次
        metadata={"help": "How many times to go through the training data."}
    )


    # ============================================
    # 3. BATCH SETTINGS
    # 控制每次处理多少训练数据
    # ============================================

    per_device_train_batch_size: int = field(
        default=2,
        # 每张 GPU 每次真正送进去 2 个样本
        # batch 越大 → GPU 显存占用越大
        metadata={"help": "How many problems to process at once."}
    )

    gradient_accumulation_steps: int = field(
        default=32,
        # 连续计算 32 个小 batch 的梯度
        # 然后才真正执行一次 optimizer update
        #
        # 用途：
        # GPU 放不下大 batch 时，
        # 用多个小 batch 模拟一个大 batch
        metadata={"help": "Accumulate gradients over multiple batches."}
    )

    # 简单理解：
    #
    # 每次 GPU 放 2 个问题
    #        ↓
    # 连续处理 32 次
    #        ↓
    # 累积梯度
    #        ↓
    # 更新一次模型
    #
    # Effective batch size:
    # 2 × 32 = 64
    #
    # 注意：
    # 如果使用多个 GPU，还需要再乘 GPU 数量


    # ============================================
    # 4. LEARNING SETTINGS
    # 控制模型每次更新多少
    # ============================================

    learning_rate: float = field(
        default=5e-6,
        # 5e-6 = 0.000005
        #
        # learning rate 越大：
        # 参数每次更新幅度越大
        #
        # 太大 → 训练可能不稳定
        # 太小 → 学习速度太慢
        metadata={"help": "How big of a step to take when learning."}
    )


    # ============================================
    # 5. GRPO SPECIFIC SETTINGS
    # GRPO 特有参数
    # ============================================

    num_generations: int = field(
        default=12,
        # 对每一个数学题生成 12 个不同回答
        #
        # 例如：
        # question
        #   ↓
        # response 1
        # response 2
        # response 3
        # ...
        # response 12
        #
        # 然后分别计算 reward
        # 再在这一组回答内部比较好坏
        #
        # 这是 GRPO 非常核心的参数
        metadata={"help": "How many different answers to generate per question."}
    )

    temperature: float = field(
        default=0.8,
        # 控制生成答案时的随机程度
        #
        # temperature 较低
        # → 输出更稳定、更确定
        #
        # temperature 较高
        # → 输出更多样
        #
        # GRPO 需要同一道题生成不同答案进行比较，
        # 所以通常需要一定随机性
        metadata={"help": "Controls randomness."}
    )

    max_new_tokens: int = field(
        default=400,
        # 模型每个回答最多生成 400 个新 token
        #
        # 数学题需要推理步骤，
        # 所以不能设置得太短
        metadata={"help": "Maximum length of generated answers."}
    )


    # ============================================
    # 6. DATA SETTINGS
    # 数据集相关参数
    # ============================================

    max_prompt_length: int = field(
        default=512,
        # 输入数学题最多允许 512 tokens
        #
        # 超过长度的 prompt 可能会被截断
        metadata={"help": "Maximum length of input questions in tokens."}
    )

    train_split_ratio: float = field(
        default=0.8,
        # 80% 数据用于训练
        # 剩余 20% 用于 validation
        #
        # train : validation = 8 : 2
        metadata={"help": "What fraction of data to use for training."}
    )


    # ============================================
    # 7. MONITORING SETTINGS
    # 控制训练过程中多久评估 / 保存 / 打印一次
    # ============================================

    eval_steps: int = field(
        default=20,
        # 每训练 20 steps
        # 做一次 evaluation
        metadata={"help": "Evaluate model every N steps."}
    )

    save_steps: int = field(
        default=20,
        # 每 20 steps 保存一个 checkpoint
        #
        # 如果训练中断，
        # 可以从 checkpoint 恢复
        metadata={"help": "Save a checkpoint every N steps."}
    )

    logging_steps: int = field(
        default=20,
        # 每 20 steps 记录一次训练指标
        #
        # 例如：
        # loss / reward / learning rate 等
        metadata={"help": "Log training metrics every N steps."}
    )

    save_total_limit: int = field(
        default=3,
        # 最多只保留最近 3 个 checkpoint
        #
        # 避免 checkpoint 太多，占满磁盘
        metadata={"help": "Keep only the N most recent checkpoints."}
    )


    # ============================================
    # 8. OTHER SETTINGS
    # 其他设置
    # ============================================

    seed: int = field(
        default=42,
        # 固定随机种子
        # 让实验尽量可以复现
        metadata={"help": "Random seed for reproducibility."}
    )

    use_8bit: bool = field(
        default=False,
        # 是否用 8-bit 加载模型
        #
        # True：
        # 显存占用更低
        #
        # False：
        # 正常精度加载
        metadata={"help": "Load model in 8-bit mode to save memory."}
    )


print("✅ Configuration class defined!")

2. field(...) 是什么？

你前面有：

from dataclasses import dataclass, field

所以这里的 field() 是 dataclasses 提供的函数。

它的作用就是：

更详细地定义 dataclass 里面的一个变量。

metadata={"help": "..."}

意思就是：

给这个参数额外挂一条说明信息。

In [ ]:
# ============================================
# Define Training Configuration Class
# ============================================

@dataclass
class TrainingConfig:
    """
    Configuration for GRPO training.

    This class holds all the settings (hyperparameters) for training.
    Think of it as a control panel with all the knobs and switches.

    Each parameter has:
    - A default value (recommended value)
    - A description (what it does)
    - A type (what kind of value it expects)
    """

    # ========== MODEL SETTINGS ==========
    # Which model to use and where to save it

    model_name: str = field(
        default="/app/models/deepseek-math-7b-base",
        metadata={"help": "The pre-trained model to start with."}
    )

    output_dir: str = field(
        default="./grpo_finetuned_model",
        metadata={"help": "Where to save the trained model. Like a 'Save As' location."}
    )

    # ========== TRAINING DURATION ==========
    # How long to train for

    num_train_epochs: int = field(
        default=5,
        metadata={"help": "How many times to go through the training data. More = more learning."}
    )

    # ========== BATCH SETTINGS ==========
    # How many examples to process at once

    per_device_train_batch_size: int = field(
        default=2,
        metadata={"help": "How many problems to process at once. Limited by GPU memory."}
    )

    gradient_accumulation_steps: int = field(
        default=32,
        metadata={"help": "Accumulate gradients over multiple batches. Simulates larger batch size."}
    )
    # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps = 64

    # ========== LEARNING SETTINGS ==========
    # How fast the model learns

    learning_rate: float = field(
        default=5e-6,  # 0.000005 in decimal
        metadata={"help": "How big of a step to take when learning. Too high = unstable."}
    )

    # ========== GRPO SPECIFIC SETTINGS ==========
    # Settings unique to GRPO algorithm

    num_generations: int = field(
        default=12,
        metadata={"help": "How many different answers to generate per question. More = better comparison."}
    )

    temperature: float = field(
        default=0.8,
        metadata={"help": "Controls randomness. 0.0 = always same answer, 1.0 = very random."}
    )

    max_new_tokens: int = field(
        default=400,
        metadata={"help": "Maximum length of generated answers. Needs to be long enough for full solutions."}
    )

    # ========== DATA SETTINGS ==========
    # How to handle the dataset

    max_prompt_length: int = field(
        default=512,
        metadata={"help": "Maximum length of input questions in tokens."}
    )

    train_split_ratio: float = field(
        default=0.8,
        metadata={"help": "What fraction of data to use for training (rest for validation)."}
    )

    # ========== MONITORING SETTINGS ==========
    # How often to check progress

    eval_steps: int = field(
        default=20,
        metadata={"help": "Evaluate model every N steps to check progress."}
    )

    save_steps: int = field(
        default=20,
        metadata={"help": "Save a checkpoint every N steps (for recovery if training stops)."}
    )

    logging_steps: int = field(
        default=20,
        metadata={"help": "Log training metrics every N steps."}
    )

    save_total_limit: int = field(
        default=3,
        metadata={"help": "Keep only the N most recent checkpoints to save disk space."}
    )

    # ========== OTHER SETTINGS ==========

    seed: int = field(
        default=42,
        metadata={"help": "Random seed for reproducibility."}
    )

    use_8bit: bool = field(
        default=False,
        metadata={"help": "Load model in 8-bit mode to save memory (slightly less accurate)."}
    )

print("✅ Configuration class defined!")

✅ Configuration class defined!


# ============================================
# Step 2: Create and Display Configuration
#
# 作用：
# 1. 根据 TrainingConfig 创建一个实际的配置对象 config
# 2. 读取并打印所有重要超参数
# 3. 检查 GRPO 训练配置是否符合预期
# ============================================


# 创建 TrainingConfig 的一个实例
config = TrainingConfig()

# 因为这里没有传入任何参数，
# 所以全部使用 TrainingConfig 中定义的默认值
#
# 例如：
# config.num_train_epochs = 5
# config.num_generations = 12
# config.learning_rate = 5e-6


# ============================================
# 1. 打印标题
# ============================================

print("TRAINING CONFIGURATION")
print("=" * 50)

# "=" * 50：
# 把 "=" 重复 50 次
# 只是为了让输出格式更清楚


# ============================================
# 2. Model Settings
# ============================================

print("\nModel Settings:")

# \n 表示换行

print(f"  Model: {config.model_name}")
# 读取 config 中的 model_name

print(f"  Output directory: {config.output_dir}")
# 读取模型保存位置


# ============================================
# 3. Training Settings
# ============================================

print("\nTraining Duration:")

print(f"  Epochs: {config.num_train_epochs}")
# 整个 training set 训练多少遍

print(f"  Batch size per device: {config.per_device_train_batch_size}")
# 每张 GPU 每个小 batch 处理多少样本

print(f"  Gradient accumulation: {config.gradient_accumulation_steps}")
# 累积多少个小 batch 后才真正更新一次参数


print(
    f"  Effective batch size: "
    f"{config.per_device_train_batch_size * config.gradient_accumulation_steps}"
)

# Effective batch size
# = batch_size × gradient_accumulation_steps
#
# 这里：
# 2 × 32 = 64
#
# 即累计约 64 个样本的梯度后更新一次参数
# （单 GPU 情况下）


# ============================================
# 4. GRPO Settings
# ============================================

print("\nGRPO Settings:")

print(f"  Generations per prompt: {config.num_generations}")
# 每道题生成多少个候选回答
# 这里 = 12

print(f"  Temperature: {config.temperature}")
# 控制生成随机性 / 多样性

print(f"  Max new tokens: {config.max_new_tokens}")
# 每个回答最多生成多少 token

print(f"  Learning rate: {config.learning_rate}")
# 模型每次参数更新的步长


# ============================================
# 5. Monitoring Settings
# ============================================

print("\nMonitoring:")

print(f"  Evaluate every: {config.eval_steps} steps")
# 每多少个训练 step 做一次 evaluation

print(f"  Save every: {config.save_steps} steps")
# 每多少 step 保存 checkpoint

print(f"  Log every: {config.logging_steps} steps")
# 每多少 step 打印 / 保存训练日志


# ============================================
# 6. 简单说明 GRPO 训练规模
# ============================================

print("\nEstimated Training Info:")

print(
    f"  This configuration will generate "
    f"{config.num_generations} answers per question"
)
# 每个问题会生成 12 个不同回答

print(f"  The model will learn by comparing these answers")
# GRPO 根据这些回答的 reward 做组内比较，
# 然后决定哪些回答应该被鼓励

In [ ]:
# ============================================
# Create and Display Configuration
# ============================================

# Create an instance of your configuration
# This uses all the default values defined above
config = TrainingConfig()

# Display all configuration values
print("TRAINING CONFIGURATION")
print("="*50)

# Group settings by category for easier reading
print("\nModel Settings:")
print(f"  Model: {config.model_name}")
print(f"  Output directory: {config.output_dir}")

print("\nTraining Duration:")
print(f"  Epochs: {config.num_train_epochs}")
print(f"  Batch size per device: {config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {config.gradient_accumulation_steps}")
print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")

print("\nGRPO Settings:")
print(f"  Generations per prompt: {config.num_generations}")
print(f"  Temperature: {config.temperature}")
print(f"  Max new tokens: {config.max_new_tokens}")
print(f"  Learning rate: {config.learning_rate}")

print("\nMonitoring:")
print(f"  Evaluate every: {config.eval_steps} steps")
print(f"  Save every: {config.save_steps} steps")
print(f"  Log every: {config.logging_steps} steps")

# Calculate approximate training time
print("\nEstimated Training Info:")
print(f"  This configuration will generate {config.num_generations} answers per question")
print(f"  The model will learn by comparing these answers")

TRAINING CONFIGURATION

Model Settings:
  Model: /app/models/deepseek-math-7b-base
  Output directory: ./grpo_finetuned_model

Training Duration:
  Epochs: 5
  Batch size per device: 2
  Gradient accumulation: 32
  Effective batch size: 64

GRPO Settings:
  Generations per prompt: 12
  Temperature: 0.8
  Max new tokens: 400
  Learning rate: 5e-06

Monitoring:
  Evaluate every: 20 steps
  Save every: 20 steps
  Log every: 20 steps

Estimated Training Info:
  This configuration will generate 12 answers per question
  The model will learn by comparing these answers


## Load the GSM8K Dataset <a id="loadGSM8K"></a>

Here you will load the GSM8K dataset, which you are already familiar with from the previous assignments.

In [ ]:
# ============================================
# Load and Explore GSM8K Dataset
# ============================================

# Load the GSM8K dataset
# This function:
# 1. Downloads the dataset (if not already downloaded)
# 2. Shows dataset statistics
# 3. Displays sample problems
dataset = load_and_explore_gsm8k_dataset()

# The dataset has two parts:
# - 'train': Problems for training (about 7,500)
# - 'test': Problems for testing (about 1,300)

Dataset Structure:
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

Dataset splits:
- Train: 7473 examples
- Test: 1319 examples

Sample Problem from Training Set:

📝 Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

✅ Answer:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

🔢 Numerical Answer: 72


## Create the Reward Function <a id="createrewardfunction"></a>

How It Works

1. **Extract** the numerical answer from the text
2. **Compare** with the correct answer
3. **Check** for partial credit (showing work, being close)
4. **Assign** a reward score

self.logger = logging.getLogger(__name__) 的作用是在 Reward Model 对象里面创建一个日志记录器，用来记录 reward 计算过程中的信息，方便调试；它不参与 reward 计算，也不参与 GRPO 的梯度更新。

创建 reward 对象
        |
        ↓
自动调用 __init__()
        |
        ↓
创建 logger
        |
        ↓
保存到 reward.logger


logging 是什么？

Python 自带的日志系统。

作用：

记录程序运行状态。

类似：

程序开始
加载模型
计算reward
出现错误
训练结束

In [ ]:
# GSM8KRewardSignal class definition
class GSM8KRewardSignal:
    """
    BASELINE Reward Model for GSM8K (Simplified Version)

    This is a basic reward model with only 3 categories:
    - Correct answer: 1.0
    - Wrong answer: 0.0
    - Unparseable: 0.0

    ⚠️ LIMITATIONS: This simple reward signal makes GRPO training difficult!
    The lack of partial credit means the model gets no signal for improvements.
    """

    def __init__(self):
        self.logger = logging.getLogger(__name__)

### Exercise 1: Extract Numerical Answer from Model Response

The first critical component of your reward model is the ability to extract numerical answers from the model's generated text. Language models often produce verbose responses with explanations, calculations, and natural language, but you need to identify and extract the actual numerical answer to compare it with the ground truth.

Currently, the basic implementation only looks for answers in the GSM8K standard format (marked with ####). However, models don't always follow this format perfectly. Your task is to implement a more robust extraction system that can handle various answer formats that models might produce.

Consider implementing patterns to match common answer phrases like "The answer is X", "equals X", "total is X", or "Therefore, X". Remember to handle edge cases such as numbers with commas (e.g., 1,000), dollar signs, negative numbers, and decimal points (you can use this regular expression: `r'([+-]?\d+(?:,\d{3})*(?:\.\d+)?)'`). As a fallback strategy, you might want to extract the last number mentioned in the response, but make sure your regex pattern is sophisticated enough to properly identify valid numbers.

输入：
The answer is $1,234.56
匹配过程：

The answer is  ✅
空格           ✅
$              ✅
1              ✅
,234           ✅
.56            ✅
返回：
["1,234.56"]

一句话总结：

(?:提示词)\s*\$?(数字)

前半部分：找到答案所在位置
后半部分：提取真正数字
()里的数字就是最后 findall() 返回的结果。



所以整体：

patterns = [
    格式1,
    格式2,
    格式3,
    格式4
]

相当于建立一个答案格式库：

可能的LLM回答方式：
比如：The answer is X
        ↓
       提取X


re.findall() 是 Python 正则表达式模块 re 中用于查找所有匹配内容的函数。

简单理解：给它一个「查找规则(pattern)」和一段「文本(text)」，它会返回所有符合规则的内容组成的 list。

基本语法
re.findall(pattern, string, flags)

参数	含义
pattern	正则表达式规则（想找什么）
string	要搜索的文本
flags	匹配模式（可选）

返回：
list




import re
text = "I have 3 apples and 5 oranges"
result = re.findall(r"\d+", text)
print(result)

输出：
['3', '5']

解释：
\d → 一个数字
+ → 一个或多个


该函数用于从 LLM 生成的数学回答文本中自动提取最终数值答案，优先匹配 GSM8K 标准格式（#### answer）和常见答案表达（如 The answer is 42、= 42 等）。
如果精确格式无法匹配，则通过正则搜索文本中的数字并选择最后一个数字作为兜底答案，最终将提取结果转换为 float，用于后续 reward 计算和模型评估。

In [ ]:
# GRADED CELL: exercise 1

def extract_numerical_answer(self, text: str) -> Optional[float]:
    """
    从模型生成的回答文本中提取最终数值答案。

    支持以下几种常见格式：
    - #### 42 （GSM8K 数据集常用格式）
    - "The answer is 42"
    - "= 42"
    - 如果以上格式都无法匹配，则寻找文本中的最后一个数字

    Args:
        text: 模型生成的完整回答文本

    Returns:
        提取出的数字(float)，如果没有找到数字则返回 None
    """

    # ==========================================
    # 第一步：检查 GSM8K 特殊答案格式
    # GSM8K 数据集中通常要求模型最后输出：
    # #### 42
    # 其中 #### 后面的数字就是最终答案。
    #
    # 例如：
    # "5 + 3 = 8\n#### 8"
    #
    # 直接提取 #### 后面的内容即可。
    # ==========================================
    if "####" in text:

        # 根据 #### 分割文本，并取最后一部分
        answer = text.split("####")[-1].strip()

        # 清理格式：
        # "$1,000" -> "1000"
        answer = answer.replace(',', '').replace('$', '')

        try:
            # 将字符串转换成浮点数
            return float(answer)

        except:
            # 如果转换失败，继续尝试其他方法
            pass


    # ==========================================
    # 第二步：定义常见答案表达方式的正则模式
    #
    # 不同模型输出答案的习惯不同，例如：
    #
    # The answer is 42
    # Answer: 42
    # x = 42
    # Total is 42
    # Therefore, 42
    #
    # 这里提前定义多个 pattern，
    # 用来匹配这些不同形式。
    #
    # 正则中的：
    # ([+-]?\d+...)
    # 表示真正需要提取的数字部分。
    # ==========================================
    patterns = [
        r"(?:The answer is|answer:|Answer:)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"(?:equals?|=)\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"(?:total|sum|result|Total|Final answer)\s*(?:is|:|=)?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"Therefore,?\s*\$?([+-]?\d+(?:,\d{3})*(?:\.\d+)?)",
    ]


    ### START CODE HERE ###

    # ==========================================
    # 第三步：依次尝试每一种答案格式
    #
    # 对每一个 pattern：
    #
    # 1. 在模型输出文本中搜索匹配内容
    # 2. 如果找到数字：
    #       - 去除逗号、美元符号
    #       - 转换成 float
    #       - 返回答案
    #
    # 例如：
    #
    # text:
    # "The answer is 25"
    #
    # 匹配：
    # "25"
    #
    # 返回：
    # 25.0
    # ==========================================
    for pattern in patterns:

        # 使用正则表达式寻找符合当前模式的答案
        matches = re.findall(
            pattern,
            text,
            re.IGNORECASE | re.MULTILINE
        )

        # 如果找到匹配结果
        if matches:
            try:

                # 取匹配到的数字
                # 去除格式符号
                answer = matches[0].replace(',', '').replace('$', '')

                # 转换为浮点数并返回
                return float(answer)

            except:

                # 如果转换失败，继续尝试下一个 pattern
                continue


    # ==========================================
    # 第四步：备用方案（Fallback）
    #
    # 如果前面的答案格式都没有匹配成功，
    # 则直接寻找文本中的所有数字。
    #
    # 例如：
    #
    # "John has 5 apples and buys 3 more.
    #  The answer is 8."
    #
    # 找到：
    # ["5", "3", "8"]
    #
    # 通常最后出现的数字是最终答案，
    # 所以选择最后一个数字。
    # ==========================================

    # 当前正则只匹配整数
    # 可以进一步扩展支持：
    # - 负数 (-5)
    # - 小数 (3.14)
    numbers = re.findall(r'\d+', text)

    if numbers:
        try:

            # 取最后出现的数字作为答案
            return float(numbers[-1].replace(',', ''))

        except:
            pass


    ### END CODE HERE ###


    # 如果所有方法都无法找到答案，
    # 返回 None
    return None


# 将该函数动态添加到 GSM8KRewardSignal 类中
# 这样该类实例可以直接调用：
#
# signal.extract_numerical_answer(text)
#
GSM8KRewardSignal.extract_numerical_answer = extract_numerical_answer

In [ ]:
# ============================================
# UNIT TEST: Exercise 1 - Extract Numerical Answer
# ============================================

def test_extract_numerical_answer():
    """Unit test for the extract_numerical_answer method."""
    print("🧪 Testing Exercise 1: Extract Numerical Answer")
    print("="*50)

    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()

    # Test cases: (input_text, expected_output, description)
    test_cases = [
        # Standard GSM8K format (already works in base implementation)
        ("The calculation is 5 + 3 = 8 #### 8", 8.0, "GSM8K standard format"),

        # Common answer phrases (MUST work after student implementation)
        ("The answer is 42", 42.0, "Common phrase: 'The answer is'"),
        ("Answer: 100", 100.0, "Common phrase: 'Answer:'"),
        ("equals 25", 25.0, "Common phrase: 'equals'"),
        ("total is 15.5", 15.5, "Common phrase: 'total is'"),
        ("Therefore, 7", 7.0, "Common phrase: 'Therefore,'"),

        # Numbers with formatting (MUST handle these)
        ("The answer is $1,234.56", 1234.56, "Dollar sign and commas"),
        ("Total: -45", -45.0, "Negative number"),
        ("equals 0.003", 0.003, "Small decimal"),

        # Fallback to last number (minimum requirement)
        ("First we have 10, then 20, finally 30", 30.0, "Last number fallback"),
        ("No clear answer but mentions 99", 99.0, "Last number in text"),
    ]

    passed = 0
    failed = 0

    for text, expected, description in test_cases:
        try:
            result = test_model.extract_numerical_answer(text)

            if result is None and expected is not None:
                print(f"❌ FAILED: {description}")
                print(f"   Input: '{text}'")
                print(f"   Expected: {expected}, Got: None")
                failed += 1
            elif result is not None and expected is None:
                print(f"❌ FAILED: {description}")
                print(f"   Input: '{text}'")
                print(f"   Expected: None, Got: {result}")
                failed += 1
            elif result is not None and abs(result - expected) > 0.01:
                print(f"❌ FAILED: {description}")
                print(f"   Input: '{text}'")
                print(f"   Expected: {expected}, Got: {result}")
                failed += 1
            else:
                print(f"✅ PASSED: {description}")
                passed += 1

        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1

    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")

    if failed == 0:
        print("🎉 All tests passed! Exercise 1 is complete.")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements:")
        print("- Extract answers from common phrases (The answer is, equals, total is)")
        print("- Handle numbers with commas and dollar signs")
        print("- Support negative numbers and decimals")
        print("- Fallback to extracting the last number in text")

    return passed == len(test_cases)

# Run the test
test_extract_numerical_answer()

🧪 Testing Exercise 1: Extract Numerical Answer
✅ PASSED: GSM8K standard format
✅ PASSED: Common phrase: 'The answer is'
✅ PASSED: Common phrase: 'Answer:'
✅ PASSED: Common phrase: 'equals'
✅ PASSED: Common phrase: 'total is'
✅ PASSED: Common phrase: 'Therefore,'
✅ PASSED: Dollar sign and commas
✅ PASSED: Negative number
✅ PASSED: Small decimal
✅ PASSED: Last number fallback
✅ PASSED: Last number in text

Results: 11/11 passed, 0/11 failed
🎉 All tests passed! Exercise 1 is complete.


True

### Exercise 2: Analyze Response Quality Indicators

Before assigning rewards to model responses, you need to understand what makes a good mathematical solution. This exercise focuses on building a comprehensive quality analysis system that examines various aspects of the generated response.

Your task is to implement quality indicators that can detect whether a response contains mathematical reasoning. The current implementation has basic checks, but you should enhance it to identify mathematical operations (not just symbols like +, -, *, / but also words like "multiply", "divide", "add", "subtract"), reasoning indicators (words like "first", "then", "next", "finally" that show step-by-step thinking), and structured solutions (numbered steps or bullet points).

Additionally, consider counting meaningful metrics like the number of sentences (which might indicate detailed explanation), checking for the presence of numbers (essential for math problems), and measuring response length. These quality indicators will be crucial for the reward functions in the following exercises, as they help distinguish between different types of responses even when you can't extract a final answer.

这一部分的意义是设计一套质量评估指标（quality indicators），从数学推理、结构化表达、答案完整性等多个角度判断 LLM 生成回答的质量，为后续 reward function 提供更全面的奖励信号。

LLM生成回答
        ↓
Quality Indicators检测
        ↓
判断：
- 是否有数学运算
- 是否有推理步骤
- 是否结构清晰
- 是否包含数字
- 回答长度是否合理
        ↓
生成reward依据
        ↓
用于强化学习优化模型

sentences = [
    s.strip()
    for s in re.split(r'[.!?]+', response)
    if s.strip()
]

等价于：

sentences = []
parts = re.split(r'[.!?]+', response)
for s in parts:
    s = s.strip()
    if s:
        sentences.append(s)


举一个最简单的例子：
import re
response = "I have 5 apples. I buy 3 more! The answer is 8?"
sentences = re.split(r'[.!?]+', response)
print(sentences)

运行结果：
[
 'I have 5 apples',
 ' I buy 3 more',
 ' The answer is 8',
 ''
]

In [ ]:
# GRADED CELL: exercise 2

def analyze_response_quality(self, response: str) -> Dict[str, any]:
    """
    分析模型回答的质量指标。

    该函数不会判断最终答案是否正确，
    而是从多个角度分析回答是否包含高质量数学推理特征。

    检查内容包括：
    - 是否包含数学计算过程
    - 是否包含多个推理步骤
    - 是否包含表示推理关系的词语
    - 是否包含数字
    - 回答长度
    - 句子数量

    Args:
        response:
            模型生成的回答文本

    Returns:
        一个包含质量指标的字典：
        - has_calculation:
            是否包含数学运算
        - has_steps:
            是否体现多步推理
        - has_reasoning:
            是否包含推理词
        - has_numbers:
            是否包含数字
        - response_length:
            回答字符长度
        - sentence_count:
            句子数量
    """

    ### START CODE HERE ###


    # ==========================================
    # 第一部分：检查是否包含数学计算
    #
    # 通过寻找数学符号或者数学相关单词，
    # 判断回答是否进行了计算。
    #
    # 例如：
    #
    # "5 + 3 = 8"
    #
    # 包含：
    # +
    # =
    #
    # 或者：
    #
    # "multiply 5 by 3"
    #
    # 包含：
    # multiply
    #
    # 注意：
    # 使用 response.lower() 可以统一大小写，
    # 例如：
    #
    # "Add" 和 "add"
    #
    # 可以被认为相同。
    # ==========================================
    calculation_indicators = [
        '=',
        '+',
        '-',
        '*',
        '/',
        'multiply',
        'divide',
        'add',
        'subtract'
    ]

    # 根据上面的指标判断回答是否包含计算内容
    # 如果任意一个指标出现在 response 中，
    # 则认为包含数学计算
    has_calculation = any(
    indicator in response.lower()
    for indicator in calculation_indicators)



    # ==========================================
    # 第二部分：检查是否包含多个步骤
    #
    # 数学推理通常是分步骤完成的。
    #
    # 判断依据：
    #
    # 1. 是否包含多个句子
    #
    # 例如：
    #
    # "First calculate 5+3.
    #  Then get the answer."
    #
    # 有两个句子。
    #
    #
    # 2. 是否明确出现 step 这样的步骤标记
    #
    # 例如：
    #
    # Step 1:
    # Step 2:
    #
    # ==========================================

    # 使用正则根据句号、问号、感叹号切分句子
    #
    # 例如：
    #
    # "Calculate first. Then add."
    #
    # 会被分成：
    #
    # [
    #   "Calculate first",
    #   "Then add"
    # ]



    sentences = [
        s.strip()
        for s in re.split(r'[.!?]+', response)
        if s.strip()
    ]

    # 统计句子数量
    sentence_count = len(sentences)

    # 判断是否存在多步推理
    # 条件：
    # - 至少两个句子
    # 或者
    # - 包含 "step" 关键词
    has_steps = sentence_count >= 2 or "step" in response.lower()



    # ==========================================
    # 第三部分：检查是否包含推理关键词
    #
    # 一些词可以表示模型正在进行逻辑推理，
    # 例如：
    #
    # therefore  因此
    # because    因为
    # first      首先
    # then       然后
    # finally    最后
    #
    # 这些词通常说明回答不是简单给结果，
    # 而是包含解释过程。
    # ==========================================

    reasoning_words = [
        'therefore',
        'because',
        'since',
        'so',
        'thus',
        'hence',
        'first',
        'then',
        'next',
        'finally',
        'must',
        'need'
    ]

    # 判断 response 中是否出现任意推理关键词
    # 使用 any() 检查：
    #
    # 只要有一个词出现，
    # 就认为包含 reasoning
    has_reasoning = any(
    word in response.lower()
    for word in reasoning_words)



    # ==========================================
    # 第四部分：检查是否包含数字
    #
    # 数学问题通常需要数字信息。
    #
    # 使用正则：
    #
    # \d
    #
    # 检查是否存在任意数字字符。
    #
    # 例如：
    #
    # "The answer is 42"
    #
    # 包含数字 4 和 2
    # ==========================================
    has_numbers = bool(re.search(r'\d', response))



    # ==========================================
    # 第五部分：统计回答长度
    #
    # 用字符数量衡量回答规模。
    #
    # 较长回答可能包含更多解释，
    # 但长度本身不是质量保证。
    # ==========================================
    response_length = len(response)


    ### END CODE HERE ###


    # 将所有质量指标保存为字典
    quality = {
        'has_calculation': has_calculation,
        'has_steps': has_steps,
        'has_reasoning': has_reasoning,
        'has_numbers': has_numbers,
        'response_length': response_length,
        'sentence_count': sentence_count
    }

    return quality


# 将该函数添加到 GSM8KRewardSignal 类中
# 之后可以通过：
#
# signal.analyze_response_quality(response)
#
# 调用该质量分析方法
GSM8KRewardSignal.analyze_response_quality = analyze_response_quality

In [ ]:
# ============================================
# UNIT TEST: Exercise 2 - Analyze Response Quality
# ============================================

def test_analyze_response_quality():
    """Unit test for the analyze_response_quality method."""
    print("🧪 Testing Exercise 2: Analyze Response Quality")
    print("="*50)

    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()

    # Test cases with expected quality indicators
    test_cases = [
        # Test 1: Response with calculations and steps
        {
            "response": "First, let's add 5 + 3 = 8. Then multiply by 2 to get 16.",
            "description": "Response with math operations and reasoning",
            "expected": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": True,
                "has_reasoning": True,  # Changed from has_reasoning_words
                "min_length": 30,
                "min_sentences": 1
            }
        },

        # Test 2: Simple answer without work
        {
            "response": "42",
            "description": "Single number response",
            "expected": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "has_reasoning": False,
                "max_length": 10,
                "max_sentences": 1
            }
        },

        # Test 3: Detailed step-by-step solution
        {
            "response": "Step 1: Calculate 10 * 5 = 50. Step 2: Subtract 15 from 50 to get 35. Finally, divide by 7 for the answer.",
            "description": "Numbered steps with calculations",
            "expected": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": True,
                "has_reasoning": True,
                "min_length": 50,
                "min_sentences": 2
            }
        },

        # Test 4: "I don't know" response
        {
            "response": "I don't know",
            "description": "Give-up response",
            "expected": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": False,
                "has_reasoning": False,
                "max_length": 20,
                "max_sentences": 1
            }
        },

        # Test 5: Response with word-based operations
        {
            "response": "We need to multiply twelve by three and then add seven.",
            "description": "Word-based math operations",
            "expected": {
                "has_calculation": True,
                "has_steps": False,
                "has_numbers": False,  # Changed: No digit characters in response
                "has_reasoning": True,
                "min_length": 20,
                "min_sentences": 1
            }
        }
    ]

    passed = 0
    failed = 0

    for test_case in test_cases:
        response = test_case["response"]
        description = test_case["description"]
        expected = test_case["expected"]

        try:
            quality = test_model.analyze_response_quality(response)

            test_passed = True
            failures = []

            # Check boolean indicators
            for key in ["has_calculation", "has_steps", "has_numbers", "has_reasoning_words"]:
                if key in expected:
                    if quality.get(key) != expected[key]:
                        failures.append(f"{key}: expected {expected[key]}, got {quality.get(key)}")
                        test_passed = False

            # Check response length
            if "min_length" in expected and quality.get("response_length", 0) < expected["min_length"]:
                failures.append(f"response_length: expected >= {expected['min_length']}, got {quality.get('response_length', 0)}")
                test_passed = False
            if "max_length" in expected and quality.get("response_length", 0) > expected["max_length"]:
                failures.append(f"response_length: expected <= {expected['max_length']}, got {quality.get('response_length', 0)}")
                test_passed = False

            # Check sentence count
            if "min_sentences" in expected and quality.get("sentence_count", 0) < expected["min_sentences"]:
                failures.append(f"sentence_count: expected >= {expected['min_sentences']}, got {quality.get('sentence_count', 0)}")
                test_passed = False
            if "max_sentences" in expected and quality.get("sentence_count", 0) > expected["max_sentences"]:
                failures.append(f"sentence_count: expected <= {expected['max_sentences']}, got {quality.get('sentence_count', 0)}")
                test_passed = False

            if test_passed:
                print(f"✅ PASSED: {description}")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Response: '{response[:50]}...' " if len(response) > 50 else f"   Response: '{response}'")
                for failure in failures:
                    print(f"   - {failure}")
                failed += 1

        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1

    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")

    if failed == 0:
        print("🎉 All tests passed! Exercise 2 is complete.")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements:")
        print("- Detect math operations (symbols AND words like 'multiply', 'add')")
        print("- Identify reasoning words ('first', 'then', 'next', 'finally')")
        print("- Detect step indicators ('step', numbered lists)")
        print("- Count sentences and measure response length")
        print("- Check for presence of numbers")

    return passed == len(test_cases)

# Run the test
test_analyze_response_quality()

🧪 Testing Exercise 2: Analyze Response Quality
✅ PASSED: Response with math operations and reasoning
✅ PASSED: Single number response
✅ PASSED: Numbered steps with calculations
✅ PASSED: Give-up response
✅ PASSED: Word-based math operations

Results: 5/5 passed, 0/5 failed
🎉 All tests passed! Exercise 2 is complete.


True

假设模型回答：

First calculate 5 + 3 = 8.
Then check the result.
Therefore, the answer is 8.

经过：

analyze_response_quality(response)

最终输出：

{
    'has_calculation': True,
    'has_steps': True,
    'has_reasoning': True,
    'has_numbers': True,
    'response_length': 76,
    'sentence_count': 3
}

### Exercise 3: Reward Unparseable Responses

Not all model responses will contain a clear numerical answer that you can extract. Sometimes the model might provide a detailed explanation without a final answer, give up with "I don't know", or produce garbled output. The current implementation harshly gives 0.0 reward to all unparseable responses, which doesn't help the model learn what aspects of its response were good or bad.

Your task is to implement a nuanced reward system for unparseable responses. Use the quality indicators from Exercise 2 to provide graduated rewards. For instance, if a response is lengthy (over 200 characters) and contains calculations, it shows the model is attempting to solve the problem and deserves some credit (perhaps 0.2). If it has reasoning and numbers but no clear answer, it might deserve 0.1. Very short responses (under 20 characters) are likely dismissive responses like "I don't know" and should receive 0.0.

The goal is to guide the model toward better responses even when it doesn't produce a parseable answer, encouraging it to show its work and reasoning process.

解决模型回答无法提取最终答案（unparseable response）时，不能简单给 0 reward 的问题，通过 Exercise 2 的质量指标，对“不完整但有价值”的回答给予部分奖励。
Exercise 3 要求

利用 Exercise 2 得到的指标：

{
has_calculation,
has_steps,
has_reasoning,
has_numbers,
response_length,
sentence_count
}

设计一个分级 reward（graduated reward）。

In [ ]:
# GRADED CELL: exercise 3

# 该函数用于计算：
# 当模型回答无法提取最终数字答案（unparseable response）时，
# 应该给予多少 reward。
#
# 核心思想：
# 不再简单地：
#
# 无法解析答案 → reward = 0
#
# 而是根据回答质量给予部分奖励。
#
# 例如：
# - 有计算过程
# - 有推理步骤
# - 包含数字
# - 回答较长
#
# 说明模型进行了有效尝试，
# 即使没有最终答案，也应该得到一定 reward。


def compute_unparseable_reward(self, response: str, correct_answer: float,
                                quality: Dict[str, any], question: str = None) -> float:
    """
    计算无法解析答案的回答的 reward。

    如果模型没有生成明确的数字答案，
    根据回答质量指标给予 0~0.3 的部分奖励。

    Args:
        response:
            模型生成的回答文本

        correct_answer:
            正确答案（这里主要用于日志记录）

        quality:
            来自 analyze_response_quality() 的质量指标

            包括：
            - 是否包含计算
            - 是否包含步骤
            - 是否包含数字
            - 回答长度等

    Returns:
        reward:
            范围在 0.0 ~ 0.3 之间
    """


    # ==========================================
    # 从质量分析结果中读取指标
    #
    # quality 是一个字典：
    #
    # {
    #   has_calculation: True/False,
    #   has_steps: True/False,
    #   has_numbers: True/False,
    #   response_length: 数值
    # }
    #
    # 使用 get() 可以避免 key 不存在时报错。
    # 核心语法：
    # 变量 = 字典.get(key, 默认值)
    # 从字典中取出指定 key 对应的 value，如果 key 不存在，就返回默认值。
    # ==========================================

    response_length = quality.get('response_length', 0)

    has_calculation = quality.get('has_calculation', False)

    has_steps = quality.get('has_steps', False)

    has_numbers = quality.get('has_numbers', False)



    # ==========================================
    # 第一层过滤：
    # 如果回答非常短，
    # 例如：
    #
    # "I don't know"
    #
    # 说明模型基本没有尝试解决问题，
    # 不给予奖励。
    # ==========================================

    if response_length < 20:
        return 0.0



    # ==========================================
    # 初始化基础奖励
    #
    # 任何超过最低长度的回答，
    # 说明模型至少进行了尝试，
    # 所以给予一个基础 reward。
    #
    # 后面根据不同质量指标继续增加。
    # ==========================================

    reward = 0.05



    ### START CODE HERE ###


    # ==========================================
    # 根据回答质量逐步增加 reward
    #
    # 每满足一个条件，
    # reward 增加一定分数。
    #
    # 目的：
    # 鼓励模型：
    #
    # 1. 展示计算过程
    # 2. 展示推理步骤
    # 3. 使用数字信息
    # 4. 提供更完整的解释
    #
    # 最终 reward 上限为 0.3。
    # ==========================================


    # If response has calculations → reward plus 0.05
    if has_calculation:
        reward += 0.05

    # If response has steps → reward plus 0.05
    if has_steps:
        reward += 0.05

    # If response has numbers → reward plus 0.05
    if has_numbers:
        reward += 0.05

    # If response is long (>100 chars) → reward plus 0.05
    if response_length >100:
        reward += 0.05

    # If response is very long (>200 chars) → reward plus another 0.05
    if response_length >200:
        reward += 0.05


    ### END CODE HERE ###



    # ==========================================
    # 输出日志，方便观察 reward 分配情况
    #
    # 记录：
    # - 模型回答
    # - 标准答案
    # - 最终reward
    # ==========================================

    self.logger.info(f"⚪ UNPARSEABLE RESPONSE:")
    self.logger.info(f"  Response: {response[:200]}...")
    self.logger.info(f"  Expected: {correct_answer}")
    self.logger.info(f"  Reward: {reward}")



    # ==========================================
    # 限制最大 reward
    #
    # 即使满足所有条件，
    # reward 最高只能为 0.3。
    #
    # 防止无法解析的回答获得过高奖励。
    # ==========================================

    return min(0.3, reward)



# 将该函数添加到 GSM8KRewardSignal 类中
# 之后可以通过：
#
# signal.compute_unparseable_reward(...)
#
# 调用该 reward 计算函数

GSM8KRewardSignal.compute_unparseable_reward = compute_unparseable_reward

In [ ]:
# ============================================
# UNIT TEST: Exercise 3 - Compute Unparseable Reward
# ============================================

def test_compute_unparseable_reward():
    """Unit test for the compute_unparseable_reward method."""
    print("🧪 Testing Exercise 3: Compute Unparseable Reward")
    print("="*50)

    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()

    # Test cases: (response, quality_dict, expected_reward_range, description)
    test_cases = [
        # Test 1: Long response with calculations (good attempt)
        {
            "response": "Let me work through this step by step. First I need to add the numbers together, then multiply by the factor mentioned. The calculation involves several steps that I'll show below.",
            "quality": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": False,
                "response_length": 180,
                "has_reasoning_words": True,
                "sentence_count": 3
            },
            "min_reward": 0.15,
            "max_reward": 0.25,
            "description": "Long response with calculations (>200 chars)"
        },

        # Test 2: Very short dismissive response
        {
            "response": "I don't know",
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": False,
                "response_length": 12,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.0,
            "max_reward": 0.0,
            "description": "Very short response (<20 chars)"
        },

        # Test 3: Medium response with some reasoning
        {
            "response": "First calculate the total, then find the average",
            "quality": {
                "has_calculation": False,
                "has_steps": True,
                "has_numbers": False,
                "response_length": 49,
                "has_reasoning_words": True,
                "sentence_count": 1
            },
            "min_reward": 0.05,
            "max_reward": 0.15,
            "description": "Medium response with reasoning"
        },

        # Test 4: Response with numbers but no final answer
        {
            "response": "We have 10 apples and 5 oranges to work with",
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 45,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.05,
            "max_reward": 0.15,
            "description": "Has numbers but no clear answer"
        },

        # Test 5: Long detailed attempt without parseable answer
        {
            "response": "To solve this problem, I would first identify all the given values. Then I'd set up the equation properly. Next, I'd solve for the unknown variable. Finally, I'd check my answer to make sure it makes sense in the context.",
            "quality": {
                "has_calculation": False,
                "has_steps": True,
                "has_numbers": False,
                "response_length": 225,
                "has_reasoning_words": True,
                "sentence_count": 4
            },
            "min_reward": 0.1,
            "max_reward": 0.25,
            "description": "Long detailed methodology without numbers"
        }
    ]

    passed = 0
    failed = 0

    for test_case in test_cases:
        response = test_case["response"]
        quality = test_case["quality"]
        min_reward = test_case["min_reward"]
        max_reward = test_case["max_reward"]
        description = test_case["description"]

        try:
            reward = test_model.compute_unparseable_reward(
                response=response,
                correct_answer=42.0,  # Arbitrary correct answer
                quality=quality,
                question="Test question"
            )

            if min_reward <= reward <= max_reward:
                print(f"✅ PASSED: {description}")
                print(f"   Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Response length: {quality['response_length']}")
                print(f"   Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                failed += 1

        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1

    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")

    if failed == 0:
        print("🎉 All tests passed! Exercise 3 is complete.")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements:")
        print("- Long responses (>200 chars) with calculations → 0.2 reward")
        print("- Responses with reasoning and numbers → 0.1 reward")
        print("- Very short responses (<20 chars) → 0.0 reward")
        print("- Any reasonable attempt → at least 0.05 reward")

    return passed == len(test_cases)

# Run the test
test_compute_unparseable_reward()

🧪 Testing Exercise 3: Compute Unparseable Reward
✅ PASSED: Long response with calculations (>200 chars)
   Reward: 0.20 (expected: 0.15-0.25)
✅ PASSED: Very short response (<20 chars)
   Reward: 0.00 (expected: 0.00-0.00)
✅ PASSED: Medium response with reasoning
   Reward: 0.10 (expected: 0.05-0.15)
✅ PASSED: Has numbers but no clear answer
   Reward: 0.10 (expected: 0.05-0.15)
✅ PASSED: Long detailed methodology without numbers
   Reward: 0.20 (expected: 0.10-0.25)

Results: 5/5 passed, 0/5 failed
🎉 All tests passed! Exercise 3 is complete.


True

### Exercise 4: Reward High-Quality Correct Answers

When the model produces the correct answer, you want to encourage not just correctness but also good mathematical communication. A response that simply states "8" is correct but less valuable than one that shows "5 + 3 = 8" with clear reasoning steps.

Your task is to implement a bonus system on top of the base 1.0 reward for correct answers. Use the quality indicators to identify exemplary responses. For example, if the response shows clear step-by-step reasoning (has_steps is true), add a 0.2 bonus. If it provides a detailed explanation (response_length > 100 characters), add 0.1. If it uses proper mathematical reasoning words, add another 0.05.

This bonus system teaches the model that you value not just the right answer but also the problem-solving process, which is crucial for building trust in AI systems and helping users understand the solution.

LLM
 |
 | 生成回答 response
 ↓
Verifier
 |
 ├── 1. Answer Parser
 |       extract_numerical_answer()
 |       
 |       作用：
 |       从文本中提取最终数字答案
 |       例如：
 |       "The answer is 42"
 |              ↓
 |             42
 |
 |
 ├── 2. Quality Analyzer
 |       analyze_response_quality()
 |
 |       检查回答质量：
 |       - 是否有计算
 |       - 是否有推理步骤
 |       - 是否有推理关键词
 |       - 是否包含数字
 |       - 回答长度
 |
 |
 └── 3. Reward Calculator
         compute_unparseable_reward()

         如果答案无法解析：
         不直接给0 reward

         根据：
         - 计算过程
         - 推理步骤
         - 数字
         - 回答长度

         给部分奖励
         
 ↓
reward
 ↓
PPO / GRPO 更新 policy model

compute_correct_reward() 用于当模型答案正确时，根据回答质量（推理步骤、详细程度、推理词、计算过程等）在基础奖励 1.0 上增加额外 bonus，鼓励模型不仅答对，还给出高质量推理过程。

In [ ]:
# GRADED CELL: exercise 4

def compute_correct_reward(self, response: str, predicted: float,
                            correct_answer: float, quality: Dict[str, any], question: str = None) -> float:
    """
    计算模型回答正确时的 reward。

    核心思想：
    - 如果最终答案正确，给予基础奖励 1.0
    - 如果回答不仅正确，而且包含高质量推理过程，
      则额外增加 bonus reward。

    例如：
    只回答：
        "42"
    reward = 1.0

    回答：
        "First calculate...
         Therefore the answer is 42."

    reward > 1.0


    Args:
        response:
            模型生成的完整回答文本

        predicted:
            从模型回答中提取出的数字答案

        correct_answer:
            标准答案

        quality:
            analyze_response_quality() 返回的质量指标

    Returns:
        reward:
            范围约为 1.0 ~ 1.3
    """


    # ==========================================
    # 初始化基础 reward
    #
    # 因为答案已经正确：
    #
    # predicted == correct_answer
    #
    # 所以给予基础奖励 1.0。
    #
    # 后续只增加额外 bonus，
    # 鼓励模型提供更好的推理过程。
    # ==========================================

    reward = 1.0



    # ==========================================
    # 已有规则：
    #
    # 如果回答包含多个步骤：
    #
    # has_steps=True
    #
    # 说明模型不是只给结果，
    # 而是展示了解题过程。
    #
    # 增加奖励。
    # ==========================================

    if quality.get('has_steps', False):
        reward += 0.1

    ### START CODE HERE ###

    # Add +0.1 bonus for detailed explanation (response_length > 100)
    if quality['response_length'] > 100:
        reward += 0.1

    # Add +0.05 bonus for using proper reasoning words
    if quality['has_reasoning']:
        reward += 0.05

    # Add +0.05 bonus for showing calculations
    if quality['has_calculation']:
        reward += 0.05

    # Cap the maximum reward at 1.3
    reward = min(1.3, reward)

    ### END CODE HERE ###
    # ==========================================
    # 记录正确回答的信息
    #
    # 方便调试：
    # - 模型预测答案
    # - 标准答案
    # - 最终reward
    # ==========================================

    self.logger.info(f"✅ CORRECT ANSWER:")
    self.logger.info(f"  Predicted: {predicted}")
    self.logger.info(f"  Expected: {correct_answer}")
    self.logger.info(f"  Reward: {reward}")


    return reward



# 将该函数添加到 GSM8KRewardSignal 类中
# 之后可以通过：
#
# signal.compute_correct_reward(...)
#
# 调用正确答案 reward 计算方法

GSM8KRewardSignal.compute_correct_reward = compute_correct_reward

In [ ]:
# ============================================
# UNIT TEST: Exercise 4 - Compute Correct Reward
# ============================================

def test_compute_correct_reward():
    """Unit test for the compute_correct_reward method."""
    print("🧪 Testing Exercise 4: Compute Correct Reward")
    print("="*50)

    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()

    # Test cases with quality indicators and expected rewards
    # UPDATED: More strict test cases to properly test bonus implementation
    test_cases = [
        # Test 1: Perfect answer with all quality indicators - should get bonuses
        {
            "response": "Step 1: Add 5 + 3 = 8. Step 2: Multiply by 2 = 16. Therefore, the answer is 16.",
            "predicted": 16.0,
            "correct": 16.0,
            "quality": {
                "has_calculation": True,
                "has_steps": True,
                "has_reasoning": True,
                "response_length": 80,
                "sentence_count": 3
            },
            "min_reward": 1.15,  # Raised from 1.0 - must have bonuses for quality
            "max_reward": 1.3,
            "description": "Perfect answer with all quality indicators (should get bonuses)"
        },
        # Test 2: Correct but minimal response - should be exactly 1.0
        {
            "response": "16",
            "predicted": 16.0,
            "correct": 16.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_reasoning": False,
                "response_length": 2,
                "sentence_count": 1
            },
            "min_reward": 1.0,
            "max_reward": 1.0,  # Must be exactly 1.0 - no bonuses
            "description": "Correct but minimal response (no bonuses)"
        },
        # Test 3: Correct with some work but not perfect - should get partial bonuses
        {
            "response": "The calculation is 5 + 3 = 8, then 8 * 2 = 16.",
            "predicted": 16.0,
            "correct": 16.0,
            "quality": {
                "has_calculation": True,
                "has_steps": False,
                "has_reasoning": False,
                "response_length": 50,
                "sentence_count": 1
            },
            "min_reward": 1.05,  # Should get some bonus but not max
            "max_reward": 1.15,
            "description": "Correct with calculation but no steps (partial bonus)"
        },
    ]

    passed = 0
    failed = 0

    for test_case in test_cases:
        response = test_case["response"]
        predicted = test_case["predicted"]
        correct = test_case["correct"]
        quality = test_case["quality"]
        min_reward = test_case["min_reward"]
        max_reward = test_case["max_reward"]
        description = test_case["description"]

        try:
            reward = test_model.compute_correct_reward(
                response=response,
                predicted=predicted,
                correct_answer=correct,
                quality=quality,
            )

            if min_reward <= reward <= max_reward:
                print(f"✅ PASSED: {description}")
                print(f"   Reward: {reward:.3f} (expected: {min_reward:.3f} - {max_reward:.3f})")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Reward: {reward:.3f} (expected: {min_reward:.3f} - {max_reward:.3f})")
                failed += 1

        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1

    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")

    if failed == 0:
        print("🎉 All tests passed! Exercise 4 is complete.")
    else:
        print("⚠️  Some tests failed. Please review your implementation:")
        print("Expected behavior:")
        print("- Base reward: 1.0 for correct answer")
        print("- +0.1 bonus for showing steps (has_steps=True)")
        print("- +0.1 bonus for detailed explanation (>100 chars)")
        print("- +0.05 bonus for using reasoning words")
        print("- +0.05 bonus for showing calculations")
        print("- Max reward: 1.3")

    return passed == len(test_cases)

# Run the test
test_compute_correct_reward()

🧪 Testing Exercise 4: Compute Correct Reward
✅ PASSED: Perfect answer with all quality indicators (should get bonuses)
   Reward: 1.200 (expected: 1.150 - 1.300)
✅ PASSED: Correct but minimal response (no bonuses)
   Reward: 1.000 (expected: 1.000 - 1.000)
✅ PASSED: Correct with calculation but no steps (partial bonus)
   Reward: 1.050 (expected: 1.050 - 1.150)

Results: 3/3 passed, 0/3 failed
🎉 All tests passed! Exercise 4 is complete.


True

### Exercise 5: Implement Partial Credit for Wrong Answers

This is perhaps the most critical exercise for effective GRPO training. The current implementation gives 0.0 reward to all wrong answers, which means the model learns nothing from near-misses or partially correct solutions. This binary reward system (1.0 for correct, 0.0 for wrong) makes learning extremely difficult and slow.

Your task is to implement a sophisticated partial credit system based on how close the wrong answer is to being correct. Start by calculating the relative error between the predicted and correct answers. Responses within 1% of the correct answer should receive 0.9 reward (they're almost right!), within 10% should get 0.7, and within 30% should get 0.5.

Additionally, check if the answer is at least in the right order of magnitude (between 0.1x and 10x the correct answer) and give 0.3 reward if so. Any reasonable attempt should get at least 0.1 reward. Finally, add a bonus of 0.1 if the response shows work (has calculations and steps) even though the final answer is wrong - this encourages the model to show its reasoning, making it easier to debug and improve.

This graduated reward system is essential for GRPO because it provides a learning gradient - the model can learn that some wrong answers are "less wrong" than others and gradually improve toward the correct solution.

LLM response
      |
      ↓
extract_numerical_answer()
      |
      ↓
能否提取答案？
      |
      ├───────────────┬───────────────┐
      |               |               |
      ↓               ↓               ↓
正确答案          错误答案        无法解析答案
      |               |               |
      ↓               ↓               ↓
compute_correct   partial reward   unparseable reward
_reward()         (Exercise 5)     (Exercise 3)

解决“答案错误就给 0 reward”的问题，通过计算错误答案与正确答案的接近程度，给予部分奖励（partial credit），让 GRPO 获得更连续、更有效的学习信号。
Exercise 5 要求

设计 graduated reward（分级奖励）。

核心：

错误答案越接近正确答案，reward 越高。

abs() 是 Python 的绝对值函数（absolute value）。

作用：

返回一个数字距离 0 的大小，去掉正负号。

语法：

abs(number)

In [ ]:
# GRADED CELL: exercise 5

def compute_wrong_reward(self, response: str, predicted: float,
                        correct_answer: float, quality: Dict[str, any], question: str = None) -> float:
    """
    Compute partial credit for wrong answers.

    This is critical for learning! Instead of 0.0 for all wrong answers,
    give partial credit based on:
    1. How close the answer is
    2. Whether work was shown
    3. Quality of reasoning

    Args:
        response: The model's generated response
        predicted: The extracted numerical answer
        correct_answer: The correct numerical answer
        quality: Quality indicators from analyze_response_quality

    Returns:
        Reward between 0.1 and 0.9
    """

    reward = 0.0  # 初始化 reward


    # Calculate relative error
    # 计算预测答案与真实答案之间的相对误差
    if correct_answer != 0:
        relative_error = abs(predicted - correct_answer) / abs(correct_answer)
    else:
        # 防止除以0
        relative_error = abs(predicted - correct_answer)


    ### START CODE HERE ###

    # 根据预测答案距离正确答案的远近给予基础奖励

    # Within 1% error → 0.9 reward
    if relative_error < 0.01:
        reward = 0.9

    # Within 5% error → 0.7 reward
    elif relative_error < 0.05:
        reward = 0.7

    # Within 10% error → 0.5 reward
    elif relative_error < 0.1:
        reward = 0.5

    # Within 30% error → 0.3 reward
    elif relative_error < 0.3:
        reward = 0.3

    # Any attempt → minimum reward
    else:
        reward = 0.1



    # 如果回答展示了计算过程和步骤
    # 说明模型进行了有效推理尝试
    if quality["has_calculation"] and quality["has_steps"]:
        reward += 0.1

    # 如果只有计算过程，没有明显步骤
    elif quality["has_calculation"]:
        reward += 0.05



    # 如果回答较长，说明可能包含更多解释
    if quality.get('response_length', 0) > 100:
        reward += 0.05


    ### END CODE HERE ###


    # 确保任何尝试至少获得 0.1 reward
    reward = max(0.1, reward)

    # 错误答案 reward 不能超过正确答案 reward
    reward = min(0.9, reward)


    self.logger.info(f"❌ WRONG ANSWER:")
    self.logger.info(f"  Predicted: {predicted}")
    self.logger.info(f"  Expected: {correct_answer}")
    self.logger.info(f"  Reward: {reward}")


    return reward


# Add the method to GSM8KRewardSignal class for grading purposes
GSM8KRewardSignal.compute_wrong_reward = compute_wrong_reward

In [ ]:
# ============================================
# UNIT TEST: Exercise 5 - Compute Wrong Reward
# ============================================

def test_compute_wrong_reward():
    """Unit test for the compute_wrong_reward method."""
    print("🧪 Testing Exercise 5: Compute Wrong Reward")
    print("="*50)

    # Create an instance of the reward model
    test_model = GSM8KRewardSignal()

    # Test cases with various error levels
    test_cases = [
        # Test 1: Almost correct (within 1%)
        {
            "response": "The answer is 100.5",
            "predicted": 100.5,
            "correct": 100.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 19,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.85,
            "max_reward": 0.95,
            "description": "Within 1% error (0.5% off)"
        },

        # Test 2: Close but wrong (within 10%)
        {
            "response": "After calculations, I get 45",
            "predicted": 45.0,
            "correct": 50.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 28,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.3,
            "max_reward": 0.4,
            "description": "Within 10% error"
        },

        # Test 3: Moderately wrong (within 30%)
        {
            "response": "The result is 70",
            "predicted": 70.0,
            "correct": 100.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 16,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.1,  # 30% error (0.3) falls into >30% bracket (0.1 base)
            "max_reward": 0.15,
            "description": "Within 30% error"
        },

        # Test 4: Right order of magnitude
        {
            "response": "Approximately 500",
            "predicted": 500.0,
            "correct": 100.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 17,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.1,
            "max_reward": 0.35,
            "description": "Right order of magnitude (5x off)"
        },

        # Test 5: Wrong but shows work
        {
            "response": "Step 1: Add 10 + 20 = 30. Step 2: Multiply by 3 = 90. The answer is 90.",
            "predicted": 90.0,
            "correct": 60.0,
            "quality": {
                "has_calculation": True,
                "has_steps": True,
                "has_numbers": True,
                "response_length": 72,
                "has_reasoning_words": False,
                "sentence_count": 3
            },
            "min_reward": 0.15,  # 0.5 for 50% error + 0.1 for showing work
            "max_reward": 0.25,
            "description": "Wrong but shows detailed work"
        },

        # Test 6: Very wrong
        {
            "response": "The answer is 1000000",
            "predicted": 1000000.0,
            "correct": 10.0,
            "quality": {
                "has_calculation": False,
                "has_steps": False,
                "has_numbers": True,
                "response_length": 21,
                "has_reasoning_words": False,
                "sentence_count": 1
            },
            "min_reward": 0.05,
            "max_reward": 0.15,
            "description": "Very wrong answer"
        }
    ]

    passed = 0
    failed = 0

    for test_case in test_cases:
        response = test_case["response"]
        predicted = test_case["predicted"]
        correct = test_case["correct"]
        quality = test_case["quality"]
        min_reward = test_case["min_reward"]
        max_reward = test_case["max_reward"]
        description = test_case["description"]

        try:
            reward = test_model.compute_wrong_reward(
                response=response,
                predicted=predicted,
                correct_answer=correct,
                quality=quality
            )

            # Calculate actual error for debugging
            relative_error = abs(predicted - correct) / (abs(correct) + 1e-10)

            if min_reward <= reward <= max_reward:
                print(f"✅ PASSED: {description}")
                print(f"   Error: {relative_error:.2%}, Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                passed += 1
            else:
                print(f"❌ FAILED: {description}")
                print(f"   Predicted: {predicted}, Correct: {correct}, Error: {relative_error:.2%}")
                print(f"   Reward: {reward:.2f} (expected: {min_reward:.2f}-{max_reward:.2f})")
                failed += 1

        except Exception as e:
            print(f"❌ ERROR: {description}")
            print(f"   Exception: {e}")
            failed += 1

    # Summary
    print("\n" + "="*50)
    print(f"Results: {passed}/{len(test_cases)} passed, {failed}/{len(test_cases)} failed")

    if failed == 0:
        print("🎉 All tests passed! Exercise 5 is complete.")
    else:
        print("⚠️ Some tests failed. Please review your implementation.")
        print("\nMinimum requirements (partial credit system):")
        print("- Within 1% error → 0.9 reward")
        print("- Within 10% error → 0.7 reward")
        print("- Within 30% error → 0.5 reward")
        print("- Right order of magnitude (0.1x-10x) → 0.3 reward")
        print("- Any reasonable attempt → 0.1 reward")
        print("- +0.1 bonus for showing work (calculations + steps)")

    return passed == len(test_cases)

# Run the test
test_compute_wrong_reward()

🧪 Testing Exercise 5: Compute Wrong Reward
✅ PASSED: Within 1% error (0.5% off)
   Error: 0.50%, Reward: 0.90 (expected: 0.85-0.95)
✅ PASSED: Within 10% error
   Error: 10.00%, Reward: 0.30 (expected: 0.30-0.40)
✅ PASSED: Within 30% error
   Error: 30.00%, Reward: 0.10 (expected: 0.10-0.15)
✅ PASSED: Right order of magnitude (5x off)
   Error: 400.00%, Reward: 0.10 (expected: 0.10-0.35)
✅ PASSED: Wrong but shows detailed work
   Error: 50.00%, Reward: 0.20 (expected: 0.15-0.25)
✅ PASSED: Very wrong answer
   Error: 9999900.00%, Reward: 0.10 (expected: 0.05-0.15)

Results: 6/6 passed, 0/6 failed
🎉 All tests passed! Exercise 5 is complete.


True

                 LLM response
                      |
                      ↓
             compute_reward()
              （总控制函数）
                      |
       ┌──────────────┼──────────────┐
       |              |              |
       ↓              ↓              ↓

extract_numerical   analyze_quality   判断答案类型
_answer()            ()

提取答案             分析质量指标
                      |
                      |
                      ↓

              ┌───────────────┐
              |
       predicted 是否存在？
              |
   ┌──────────┼──────────┐
   |          |          |
   ↓          ↓          ↓

无答案       正确答案      错误答案

   |           |            |
   ↓           ↓            ↓

Exercise3   Exercise4    Exercise5

compute_    compute_     compute_
unparseable correct      wrong
_reward()   _reward()    _reward()

   |           |            |
   └───────────┼────────────┘
               ↓
          final reward

compute_reward() 是整个 reward system 的主入口（orchestrator），负责接收模型回答，提取答案、分析质量，然后根据回答类型调用对应的 reward 计算函数，返回最终 reward。

In [ ]:
# This cell will be graded - compute_reward method (main orchestrator)
def compute_reward(self, response: str, correct_answer: float, question: str = None) -> float:
    """
    Main reward computation function - delegates to specialized methods.

    This function orchestrates the reward computation by:
    1. Extracting numerical answer from response
    2. Analyzing response quality indicators
    3. Calling appropriate reward computation method
    4. Returning final reward value

    Args:
        response: The model's response text
        correct_answer: The correct numerical answer
        question: The original question (optional)

    Returns:
        float: Final reward value for this response
    """
    # Step 1: Try to extract numerical answer
    predicted = self.extract_numerical_answer(response)

    # Step 2: Analyze response quality
    quality = self.analyze_response_quality(response)

    # Step 3: Route to appropriate reward computation
    if predicted is None:
        # Case 1: Could not parse any numerical answer
        return self.compute_unparseable_reward(response, correct_answer, quality, question)
    elif abs(predicted - correct_answer) < 0.01:
        # Case 2: Answer is correct (within small tolerance)
        return self.compute_correct_reward(response, predicted, correct_answer, quality, question)
    else:
        # Case 3: Answer is wrong
        return self.compute_wrong_reward(response, predicted, correct_answer, quality, question)

# Add the method to GSM8KRewardSignal class for grading purposes
GSM8KRewardSignal.compute_reward = compute_reward

print("✅ Reward model class defined!")
print("This will be your 'grading system' for model responses.")

✅ Reward model class defined!
This will be your 'grading system' for model responses.


In [ ]:
# ============================================
# Test the Reward Model
# ============================================

# Test your reward model with different types of answers
print("🧪 TESTING THE REWARD MODEL")
print("="*60)
print("See how it grades different types of responses:\n")

reward_model = GSM8KRewardSignal()
print("✅ Reward model created with detailed logging for debugging!")
print("Check the log file to see question, response, and reward details for each generation.")

# Test cases: (response, correct_answer)
test_cases = [
    # Perfect answer with steps
    ("Let me calculate step by step: 5 + 3 = 8. The answer is #### 8", 8.0),

    # Correct but no steps
    ("The answer is 8", 8.0),

    # Close but wrong
    ("5 + 3 = 7. So the answer is 7.", 8.0),

    # Very wrong
    ("The total is 100", 8.0),

    # Shows work but wrong
    ("First I add 5 + 3 = 9. Then I multiply by 2 = 18. Answer: 18", 8.0),

    # No attempt
    ("I don't know how to solve this", 8.0),
]

for i, (response, correct) in enumerate(test_cases, 1):
    print(f"Test {i}:")
    print(f"  Response: \"{response[:50]}...\"" if len(response) > 50 else f"  Response: \"{response}\"")

    # Compute reward
    reward = reward_model.compute_reward(response, correct, "Test question")

    # Extract answer for display
    extracted = reward_model.extract_numerical_answer(response)

    print(f"  Extracted answer: {extracted}")
    print(f"  Correct answer: {correct}")
    print(f"  Reward: {reward:.2f}")

    # Explain the reward
    if reward >= 1.0:
        print("  ✅ Excellent!")
    elif reward >= 0.5:
        print("  🟨 Good attempt")
    elif reward >= 0.2:
        print("  🟠 Some credit")
    else:
        print("  ⚪ Minimal credit")
    print()

print(f"\n💡 Note: Detailed logs are saved to: {log_file}")

🧪 TESTING THE REWARD MODEL
See how it grades different types of responses:

✅ Reward model created with detailed logging for debugging!
Check the log file to see question, response, and reward details for each generation.
Test 1:
  Response: "Let me calculate step by step: 5 + 3 = 8. The answ..."
  Extracted answer: 8.0
  Correct answer: 8.0
  Reward: 1.15
  ✅ Excellent!

Test 2:
  Response: "The answer is 8"
  Extracted answer: 8.0
  Correct answer: 8.0
  Reward: 1.00
  ✅ Excellent!

Test 3:
  Response: "5 + 3 = 7. So the answer is 7."
  Extracted answer: 7.0
  Correct answer: 8.0
  Reward: 0.40
  🟠 Some credit

Test 4:
  Response: "The total is 100"
  Extracted answer: 100.0
  Correct answer: 8.0
  Reward: 0.10
  ⚪ Minimal credit

Test 5:
  Response: "First I add 5 + 3 = 9. Then I multiply by 2 = 18. ..."
  Extracted answer: 18.0
  Correct answer: 8.0
  Reward: 0.20
  🟠 Some credit

Test 6:
  Response: "I don't know how to solve this"
  Extracted answer: None
  Correct answer: 8.0
  

## Load the Language Model <a id="loadthelanguagemodel"></a>

### About DeepSeek Math Model

You will be using **DeepSeek Math 7B Base** Model

### What You are Loading

1. **Tokenizer**: Converts text to numbers the model understands
2. **Model**: The actual neural network with all the parameters

### Memory Requirements

- The model needs about 14-28 GB of GPU memory for loading the model
- Loading may take 1-2 minutes

加载模型对应的 tokenizer，将文本转换成模型可以理解的 token ID，并配置 padding 方式，最后测试文本分词效果。

In [ ]:
# ============================================
# Load the Tokenizer
# ============================================

print("Loading tokenizer...")
print("The tokenizer converts text to tokens (numbers) that the model understands.\n")

is_local = os.path.exists(config.model_name)

# Load the tokenizer
# trust_remote_code=True allows loading custom code from the model repository
tokenizer = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,  # Some models have custom tokenizer code
    local_files_only=is_local
)

# Set up padding token
# Padding is used to make all inputs the same length
if tokenizer.pad_token is None:
    # If no padding token defined, use the end-of-sequence token
    tokenizer.pad_token = tokenizer.eos_token

# Set padding to left side for generation tasks
# This ensures the actual text is on the right (where model expects it)
tokenizer.padding_side = "left"

print("✅ Tokenizer loaded successfully!")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")
print(f"  Padding token: {tokenizer.pad_token}")
print(f"  End-of-sequence token: {tokenizer.eos_token}")

# Example of tokenization
example_text = "What is 5 + 3?"
tokens = tokenizer.encode(example_text)
print(f"\nExample tokenization:")
print(f"  Text: \"{example_text}\"")
print(f"  Tokens: {tokens[:10]}..." if len(tokens) > 10 else f"  Tokens: {tokens}")
print(f"  Number of tokens: {len(tokens)}")

Loading tokenizer...
The tokenizer converts text to tokens (numbers) that the model understands.

✅ Tokenizer loaded successfully!
  Vocabulary size: 100,002 tokens
  Padding token: <｜end▁of▁sentence｜>
  End-of-sequence token: <｜end▁of▁sentence｜>

Example tokenization:
  Text: "What is 5 + 3?"
  Tokens: [100000, 2640, 317, 207, 20, 919, 207, 18, 30]
  Number of tokens: 9


# ============================================
# 加载语言模型（Language Model）
# ============================================

print("Loading the DeepSeek Math model...")
print("根据网络速度不同，模型加载可能需要1-2分钟。\n")



# ============================================
# 设置模型加载参数
# ============================================
#
# model_kwargs:
# 保存加载模型时需要传入的配置参数。
#
# 这些参数会传递给：
#
# AutoModelForCausalLM.from_pretrained()
#
model_kwargs = {


    # 允许加载模型仓库中自定义的模型代码
    #
    # 一些模型（例如 DeepSeek、LLaMA 部分变体）
    # 会提供自己的 modeling 文件。
    #
    # 设置 True 可以执行这些自定义代码。
    #
    "trust_remote_code": True,



    # 设置模型权重的数据类型(dtype)
    #
    # GPU:
    # 使用 bfloat16
    #
    # CPU:
    # 使用 float32
    #
    # bfloat16:
    # - 占用显存更少
    # - 计算速度更快
    # - 精度通常足够训练/推理
    #
    # float32:
    # - 精度最高
    # - 占用更多内存
    #
    "dtype": torch.bfloat16 if torch.cuda.is_available()
             else torch.float32,



    # 是否开启 KV cache
    #
    # 在文本生成时：
    #
    # Transformer 每生成一个 token，
    # 不需要重新计算之前 token 的 Key 和 Value。
    #
    # 保存之前计算结果：
    #
    # KV cache
    #
    # 可以显著提高生成速度。
    #
    "use_cache": True,



    # 如果模型已经存在本地：
    # 只从本地加载
    #
    # 如果不存在：
    # 可以从远程下载
    #
    "local_files_only": is_local,
}



# ============================================
# 可选：8-bit 量化加载
# ============================================
#
# 如果显存不足，可以使用 int8 量化。
#
# 原始模型：
#
# float16/bfloat16
#       |
#       ↓
# int8
#
# 优点：
# - 显存占用降低
# - 可以在更小GPU上运行
#
# 缺点：
# - 有轻微精度损失
#
if config.use_8bit:

    print("Using 8-bit mode to save memory...")

    # 开启8-bit量化加载
    model_kwargs["load_in_8bit"] = True

    # 大约减少50%左右显存占用
    # 对模型性能影响通常较小



# ============================================
# 加载模型
# ============================================
#
# AutoModelForCausalLM:
#
# 加载因果语言模型（Causal Language Model）
#
# 例如：
# GPT
# LLaMA
# DeepSeek
#
# 工作方式：
#
# 输入：
# "The answer is"
#
# 预测：
# "42"
#
# 即：
#
# 根据前面的 token 预测下一个 token。
#
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    **model_kwargs
)



# ============================================
# 将模型移动到指定设备
# ============================================
#
# device:
#
# 可能是：
#
# "cuda"
# GPU
#
# "cpu"
# CPU
#
# GPU:
# 训练和推理速度更快
#
model.to(device)



print("✅ Model loaded successfully!")



# ============================================
# 输出模型信息
# ============================================

print(f"\nModel Information:")



# 查看模型类型
#
# 例如：
#
# DeepSeekForCausalLM
# LlamaForCausalLM
#
print(
    f"  Model type: {model.__class__.__name__}"
)



# ============================================
# 计算模型参数数量
# ============================================
#
# model.parameters():
# 获取模型所有参数
#
# p.numel():
# 一个 tensor 中元素数量
#
# sum():
# 所有参数数量求和
#
# 除以 1e9:
# 转换成 billion（十亿）
#
# 例如：
#
# 7,000,000,000
#
# =
#
# 7B 参数
#
print(
    f"  Number of parameters: "
    f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f} billion"
)



# ============================================
# 检查模型所在设备
# ============================================

# 获取模型第一个参数
# 检查是否在 CUDA(GPU)
if next(model.parameters()).is_cuda:

    print(
        f"  Location: GPU (fast training!)"
    )

else:

    print(
        f"  Location: CPU (slower training)"
    )

In [ ]:
# ============================================
# Load the Language Model
# ============================================

print("Loading the DeepSeek Math model...")
print("This may take 1-2 minutes depending on your internet speed.\n")

# Set up model loading arguments
model_kwargs = {
    "trust_remote_code": True,  # Allow custom model code
    "dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    "use_cache": True,
    "local_files_only": is_local,
    # bfloat16: Uses less memory than float32 but maintains good precision
    # float32: Full precision (used on CPU)
}

# Optional: Use 8-bit quantization to save memory
if config.use_8bit:
    print("Using 8-bit mode to save memory...")
    model_kwargs["load_in_8bit"] = True
    # 8-bit mode reduces memory usage by ~50% with minimal accuracy loss

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    **model_kwargs
)
model.to(device)
print("✅ Model loaded successfully!")

# Display model information
print(f"\nModel Information:")
print(f"  Model type: {model.__class__.__name__}")
print(f"  Number of parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} billion")

# Check model device
if next(model.parameters()).is_cuda:
    print(f"  Location: GPU (fast training!)")
else:
    print(f"  Location: CPU (slower training)")

Loading the DeepSeek Math model...
This may take 1-2 minutes depending on your internet speed.

✅ Model loaded successfully!

Model Information:
  Model type: LlamaForCausalLM
  Number of parameters: 6.91 billion
  Location: GPU (fast training!)


**inputs

是 Python 的字典展开（dictionary unpacking）。

先看前面：

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
)

tokenizer 返回的是一个字典：

类似：

inputs = {
    "input_ids": tensor([[123, 456, 789]]),
    "attention_mask": tensor([[1, 1, 1]])
}

# ============================================
# 将输入数据移动到模型所在设备（GPU/CPU）
# ============================================
#
# tokenizer 默认生成的 tensor 在 CPU 上：
#
# inputs = {
#     "input_ids": CPU tensor,
#     "attention_mask": CPU tensor
# }
#
# 但是如果模型已经加载到了 GPU：
#
# model → CUDA
#
# 那么输入也必须移动到 GPU。
#
# 否则：
#
# 模型：
# GPU
#
# 输入：
# CPU
#
# 会产生 device mismatch 错误。
#
# 例如：
# Expected all tensors to be on the same device
#
if torch.cuda.is_available():

    # 字典推导式：
    #
    # inputs.items()
    # 会遍历字典中的 key 和 value
    #
    # 例如：
    #
    # k = "input_ids"
    # v = token tensor
    #
    # 然后：
    #
    # v.cuda()
    #
    # 将 tensor 从 CPU 移动到 GPU。
    #
    # 等价于：
    #
    # inputs["input_ids"] = inputs["input_ids"].cuda()
    # inputs["attention_mask"] = inputs["attention_mask"].cuda()
    #
    inputs = {
        k: v.cuda()
        for k, v in inputs.items()
    }

In [ ]:
# ============================================
# Enable Memory Optimizations
# ============================================

print("Enabling memory optimizations...\n")

# Enable gradient checkpointing
# This trades computation for memory by not storing all intermediate values
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled")
    print("   This saves memory by recomputing values when needed")
    print("   Training will be slightly slower but use less memory")
else:
    print("Gradient checkpointing not available for this model")

# Test the model with a simple generation
print("\nTesting model with a simple math problem...")
test_prompt = "What is 2 + 2? The answer is"
inputs = tokenizer(test_prompt, return_tensors="pt")

# Move inputs to same device as model
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

# Generate a short response
with torch.no_grad():  # Don't calculate gradients for this test
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.1,  # Low temperature for deterministic output
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id  # Explicitly set to suppress warning
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"  Prompt: \"{test_prompt}\"")
print(f"  Model response: \"{response}\"")
print("\n✅ Model is working correctly!")

Enabling memory optimizations...

✅ Gradient checkpointing enabled
   This saves memory by recomputing values when needed
   Training will be slightly slower but use less memory

Testing model with a simple math problem...
  Prompt: "What is 2 + 2? The answer is"
  Model response: "What is 2 + 2? The answer is 4. What is 2 + 2"

✅ Model is working correctly!


## Prepare Training and Validation Datasets <a id="preparetraining"></a>

### Data Splitting

You need to split the data into two parts:
1. **Training set** (80%): Used to train the model
2. **Validation set** (20%): Used to check progress during training

### Data Preparation Steps

For each problem, we:
1. Create a prompt (the question)
2. Extract the numerical answer
3. Format it for the model

### Prompt Template

You use a specific format to help the model understand what you want:
```
Question: [math problem]
Let's solve this step-by-step and find the numerical answer:
```

In [ ]:
# ============================================
# Prepare Training and Validation Datasets
# ============================================

print("Preparing datasets for training...\n")

# Use the utility function to prepare the datasets
# This function:
# 1. Loads the GSM8K training data
# 2. Splits it into train/validation
# 3. Formats each problem with the prompt template
# 4. Extracts numerical answers
train_dataset, eval_dataset = prepare_dataset(config, tokenizer)

# Display dataset statistics
print("\nDataset Statistics:")
print(f"  Total examples: {len(train_dataset) + len(eval_dataset):,}")
print(f"  Training examples: {len(train_dataset):,} ({len(train_dataset)/(len(train_dataset)+len(eval_dataset))*100:.1f}%)")
print(f"  Validation examples: {len(eval_dataset):,} ({len(eval_dataset)/(len(train_dataset)+len(eval_dataset))*100:.1f}%)")

# Calculate training iterations
steps_per_epoch = len(train_dataset) // (config.per_device_train_batch_size * config.gradient_accumulation_steps)
total_steps = steps_per_epoch * config.num_train_epochs
print(f"\nTraining Iterations:")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total training steps: {total_steps}")
print(f"  Evaluations during training: {total_steps // config.eval_steps}")

Preparing datasets for training...


Dataset Statistics:
  Total examples: 7,473
  Training examples: 5,978 (80.0%)
  Validation examples: 1,495 (20.0%)

Training Iterations:
  Steps per epoch: 93
  Total training steps: 465
  Evaluations during training: 23


In [ ]:
# ============================================
# Show Sample Training Examples
# ============================================

print("📝 Sample Training Examples:")
print("="*60)

# Show 3 examples from the training set
for i in range(min(3, len(train_dataset))):
    sample = train_dataset[i]

    print(f"\nExample {i+1}:")
    print("-"*40)

    # Show the prompt (that you give the model)
    print("PROMPT (Input to model):")
    print(sample['prompt'][:300] + "..." if len(sample['prompt']) > 300 else sample['prompt'])

    # Show the expected answer
    print(f"\nEXPECTED NUMERICAL ANSWER: {sample['answer']}")

    # Show part of the solution
    if 'answer_text' in sample:
        print("\nSOLUTION STEPS (first part):")
        print(sample['answer_text'][:200] + "..." if len(sample['answer_text']) > 200 else sample['answer_text'])

print("\n" + "="*60)
print("The model will learn to generate step-by-step solutions like these!")

📝 Sample Training Examples:

Example 1:
----------------------------------------
PROMPT (Input to model):
Question: Stefan goes to a restaurant to eat dinner with his family. They order an appetizer that costs $10 and 4 entrees that are $20 each. If they tip 20% of the total for the waiter, what is the total amount of money that they spend at the restaurant?

Let's solve this step-by-step and find the n...

EXPECTED NUMERICAL ANSWER: 108.0

SOLUTION STEPS (first part):
The total cost of the entrees is 4 * $20 = $<<4*20=80>>80.
The total cost of the dinner is $80 + $10 = $<<80+10=90>>90.
The tip is $90 * 0.20 = $<<90*0.20=18>>18
The total cost with tip is $90 + $18 =...

Example 2:
----------------------------------------
PROMPT (Input to model):
Question: The gauge on a water tank shows that the tank is 1/3 full of water. To fill the tank, 16 gallons of water are added. How many gallons of water does the tank hold when full?

Let's solve this step-by-step and find the numerical answer:

## Create Evaluation Callback <a id="createevaluation"></a>

### Why Evaluation Matters

During training, you want to know:
- Is the model getting better?
- What's the current accuracy?
- Should you stop training?

### What You Track

Our evaluation callback measures:
1. **Accuracy**: Percentage of correct answers
2. **Average Reward**: How good the answers are overall
3. **Sample Outputs**: Actual model responses

### Evaluation Frequency

You evaluate:
- Every 20 training steps (configurable)
- On the test set (never seen during training)
- Using a sample for speed

Create Evaluation Callback 总结

一句话：

Evaluation Callback 用于在 GRPO 训练过程中定期评估模型表现，监控模型是否变好，并记录 accuracy、reward 和生成样例。

# 从本地磁盘加载 GSM8K 数据集
#
# load_from_disk:
# HuggingFace datasets 提供的方法
#
# 读取之前保存好的 dataset。
#
gsm8k = load_from_disk(
    "/app/data/gsm8k"
)


# 获取 test split
#
# GSM8K 数据结构：
#
# {
#    "train": ...,
#    "test": ...
# }
#
test_dataset = gsm8k["test"]



Callback 的核心思想

简单理解：

提前注册一个函数/对象，当某个事件发生时，自动调用它。

比如训练：

step 1
step 2
step 3
...
step 20
       |
       ↓
触发 callback
       |
       ↓
evaluate model
       |
       ↓
记录 accuracy/reward

你不用自己写：

if step % 20 == 0:
    evaluate()

训练框架会自动调用 callback。

In [ ]:
# ============================================
# Set Up Evaluation System
# ============================================

print("Setting up evaluation system...\n")

# Load the test dataset for evaluation
# IMPORTANT: you should use the TEST set (not training data) to measure true performance
print("Loading GSM8K test dataset...")
gsm8k = load_from_disk("/app/data/gsm8k")
test_dataset = gsm8k["test"]
print(f"✅ Loaded {len(test_dataset):,} test examples")
print("   These are NEVER seen during training (prevents cheating)\n")

# Create the evaluation callback
# This will run periodically during training to check progress
eval_callback = GSM8KEvaluationCallback(
    tokenizer=tokenizer,
    test_dataset=test_dataset,
    batch_size=64,  # How many examples to evaluate at once
    sample_size=1.0  # Use full test set (1.0 = 100%)
)

print("✅ Evaluation system configured!")
print(f"\nEvaluation Details:")
print(f"  Will evaluate on: {len(test_dataset)} test examples")
print(f"  Evaluation frequency: Every {config.eval_steps} training steps")
print(f"  Metrics tracked: Accuracy, Average Reward, Sample Outputs")
print(f"\nTip: Watch the accuracy increase during training!")

Setting up evaluation system...

Loading GSM8K test dataset...
✅ Loaded 1,319 test examples
   These are NEVER seen during training (prevents cheating)

✅ Evaluation system configured!

Evaluation Details:
  Will evaluate on: 1319 test examples
  Evaluation frequency: Every 20 training steps
  Metrics tracked: Accuracy, Average Reward, Sample Outputs

Tip: Watch the accuracy increase during training!


## Configure GRPO Trainer <a id="configuregrpotrainer"></a>

### The GRPO Training Process

Here's how GRPO training works:

1. **Generate Multiple Answers**: For each question, generate 12 different answers
2. **Score Each Answer**: Use your reward model to grade each one
3. **Compare Within Group**: See which answers are better than others
4. **Update Model**: Teach it to prefer the better answers

Instead of saying "this answer is worth 0.7 points", GRPO says "Answer #3 is better than answers #1, #2, #4, #5...". This relative comparison is more stable and effective than scoring each answer.

### Key Components

1. **Trainer**: Orchestrates the training process
2. **Reward Function**: Grades the generated answers
3. **Configuration**: All your settings from earlier

Note the `GRPOConfig()` within the `create_grpo_trainer()` function. This is where you set various parameters such as the temperature, as you have seen in the video.

假设你的 train_dataset 长这样：

train_dataset = [
    {
        "prompt": "What is 1+1?",
        "answer": "2",
        "question": "1+1等于多少？"
    },
    {
        "prompt": "What is 2+3?",
        "answer": "5",
        "question": "2+3等于多少？"
    }
]

for item in train_dataset:
    prompt2ans[item['prompt']] = (
        item['answer'],
        item['question']
    )本质上是在做：

prompt2ans = {
    "What is 1+1?": ("2", "1+1等于多少？"),
    "What is 2+3?": ("5", "2+3等于多少？")
}

config.num_generations

意思是：

从 config 这个对象里面，取出名为 num_generations 的属性。

假设前面有：

class Config:
    def __init__(self):
        self.num_generations = 12
        self.temperature = 0.7
        self.learning_rate = 1e-5

config = Config()

train_dataset
     │
     ├──→ 建立 prompt2ans
     │       prompt → 正确答案
     │
     └──→ GRPOConfig
              │
              ├─ generation 参数
              │   ├─ num_generations
              │   ├─ temperature
              │   ├─ top_p
              │   └─ max_new_tokens
              │
              ├─ training 参数
              │   ├─ batch size
              │   ├─ gradient accumulation
              │   └─ epochs
              │
              ├─ optimizer 参数
              │   ├─ learning rate
              │   ├─ Adam β1 / β2
              │   └─ weight decay
              │
              └─ logging / save / eval

# ============================================
# 定义 GRPO Trainer 配置创建函数
# ============================================

def create_grpo_trainer(
    config,
    model,
    tokenizer,
    train_dataset,
    eval_dataset,
    reward_model,
    test_dataset=None
):
    """
    创建并配置 GRPO 训练所需要的参数。

    这个函数目前主要做两件事：
    1. 创建 prompt -> 正确答案 的快速查询字典 prompt2ans
    2. 创建 GRPOConfig，设置 GRPO 训练相关超参数

    注意：
    虽然函数名叫 create_grpo_trainer，
    但这里实际上还没有真正创建 GRPOTrainer，
    目前返回的是：
        GRPO_config
        prompt2ans

    Args:
        config:
            自己定义的训练配置对象，
            里面保存 batch size、学习率、epoch 等参数

        model:
            要进行 GRPO 训练的语言模型
            （当前函数中暂时没有直接使用）

        tokenizer:
            模型对应的 tokenizer，
            主要用于获取 pad_token_id 和 eos_token_id

        train_dataset:
            GRPO 的训练数据集

        eval_dataset:
            验证数据集
            （当前函数中暂时没有直接使用）

        reward_model:
            Reward Model / Reward Function
            用来评价模型生成答案的质量
            （当前函数中暂时没有直接使用）

        test_dataset:
            测试数据集，用于最终评估
            （当前函数中暂时没有直接使用）

    Returns:
        GRPO_config:
            GRPO 训练配置

        prompt2ans:
            prompt -> (正确答案, 原问题) 的映射字典
    """

    print("🔧 Creating GRPO trainer...\n")

    # =====================================================
    # 1. 建立 prompt -> answer 的快速查询表
    # =====================================================

    # 使用 Python 字典保存：
    #
    # prompt
    #   ↓
    # (answer, question)
    #
    # 例如：
    #
    # prompt2ans["What is 2 + 2?"]
    # =
    # ("4", "What is 2 + 2?")
    #
    # 后面计算 reward 时，
    # 可以根据模型当前回答对应的 prompt，
    # 快速找到正确答案。
    #
    # Dictionary 查询平均时间复杂度约为 O(1)，
    # 比每次遍历整个 dataset 查答案效率高很多。

    prompt2ans = {}

    for item in train_dataset:

        # item['prompt']:
        # 发送给模型的完整 prompt
        #
        # item['answer']:
        # 标准答案 / ground truth
        #
        # item['question']:
        # 原始数学题目

        prompt2ans[item['prompt']] = (
            item['answer'],
            item['question']
        )

    print(f"Built answer lookup with {len(prompt2ans):,} entries")


    # =====================================================
    # 2. 创建 GRPOConfig
    # =====================================================
    #
    # GRPOConfig 决定：
    #
    # 生成阶段：
    #   一个 prompt 生成几个答案？
    #   temperature 多大？
    #   最多生成多少 token？
    #
    # 训练阶段：
    #   batch size 多大？
    #   学习率多少？
    #   训练几个 epoch？
    #
    # 优化阶段：
    #   optimizer 参数
    #   gradient clipping
    #   weight decay
    #
    # 监控阶段：
    #   多久 logging / evaluation / save 一次？
    #
    # =====================================================

    GRPO_config = GRPOConfig(

        # =================================================
        # GRPO SPECIFIC
        # GRPO 特有配置
        # =================================================

        # 对于每一个 prompt，
        # 生成多少个候选答案（rollouts / generations）
        #
        # 例如：
        # num_generations = 12
        #
        # 一个数学问题 x：
        #
        # x
        # ├── y1
        # ├── y2
        # ├── y3
        # ├── ...
        # └── y12
        #
        # Reward function 会分别给这些答案计算 reward，
        # 然后 GRPO 会在同一组答案内部进行相对比较。
        num_generations=config.num_generations,


        # 控制生成随机性。
        #
        # temperature 越低：
        #     输出越确定
        #
        # temperature 越高：
        #     输出越多样
        #
        # GRPO 训练需要一定程度的 exploration，
        # 所以通常不能完全 greedy generation。
        temperature=config.temperature,


        # 模型生成答案时使用的具体参数
        generation_kwargs={

            # 每个回答最多可以生成多少个新 token
            #
            # 防止模型生成过长的推理过程。
            "max_new_tokens": config.max_new_tokens,


            # 再次指定采样 temperature
            "temperature": config.temperature,


            # nucleus sampling / Top-p sampling
            #
            # 只从累计概率达到 95% 的 token 中进行采样。
            #
            # 例如：
            #
            # token probabilities:
            #
            # A: 0.50
            # B: 0.25
            # C: 0.15
            # D: 0.06
            # E: 0.04
            #
            # 累计到 0.95 左右以后，
            # 后面的极低概率 token 会被排除。
            #
            # 目的：
            # 保持生成多样性的同时，
            # 避免采样到过于离谱的 token。
            "top_p": 0.95,


            # 开启随机采样。
            #
            # True:
            # temperature / top_p 会真正发挥作用
            #
            # False:
            # 通常会趋向 greedy decoding
            "do_sample": True,


            # Padding token 的 ID
            #
            # 当不同样本长度不一致时，
            # batch 中较短的样本需要 padding。
            "pad_token_id": tokenizer.pad_token_id,


            # End Of Sequence token
            #
            # 模型生成这个 token 后，
            # 可以认为当前回答生成结束。
            "eos_token_id": tokenizer.eos_token_id,
        },


        # 一次 generation 阶段最多处理多少条生成结果。
        #
        # 这里：
        #
        # batch_size × num_generations
        #
        # 比如：
        #
        # per_device_train_batch_size = 4
        # num_generations = 12
        #
        # 需要产生：
        #
        # 4 × 12 = 48
        #
        # 个 completion。
        #
        # min(96, ...)
        # 是为了限制最大 generation batch，
        # 避免显存占用过高。
        generation_batch_size=min(
            96,
            config.per_device_train_batch_size
            * config.num_generations
        ),


        # =================================================
        # OUTPUT SETTINGS
        # 输出设置
        # =================================================

        # checkpoint、日志等训练结果保存的位置
        output_dir=config.output_dir,


        # =================================================
        # TRAINING DURATION
        # 训练时间
        # =================================================

        # 整个 train_dataset 完整训练多少遍。
        #
        # 例如：
        #
        # num_train_epochs = 5
        #
        # 就是整个训练集大约重复训练 5 次。
        num_train_epochs=config.num_train_epochs,


        # =================================================
        # BATCH SETTINGS
        # Batch 相关设置
        # =================================================

        # 每张 GPU 一次 forward/backward
        # 实际送入多少个 prompt。
        #
        # 注意：
        # 在 GRPO 中，
        # 每个 prompt 还会生成 num_generations 个答案。
        per_device_train_batch_size=
            config.per_device_train_batch_size,


        # 梯度累积次数。
        #
        # 假设：
        #
        # batch_size = 2
        # gradient_accumulation_steps = 32
        #
        # 那么模型不会每 2 个样本就 optimizer.step()，
        # 而是累积 32 个小 batch 的 gradient 后再更新。
        #
        # 等效 batch 大致为：
        #
        # effective batch size
        # ≈
        # per_device_train_batch_size
        # × gradient_accumulation_steps
        # × GPU 数量
        #
        # 这样可以在显存有限时模拟更大的 batch。
        gradient_accumulation_steps=
            config.gradient_accumulation_steps,


        # =================================================
        # LEARNING SETTINGS
        # 学习率相关设置
        # =================================================

        # Optimizer 每次更新参数时的步长。
        #
        # learning rate 太大：
        #     训练可能不稳定
        #
        # learning rate 太小：
        #     学习速度慢
        learning_rate=config.learning_rate,


        # Cosine learning rate scheduler
        #
        # 学习率大致按照余弦曲线逐渐降低：
        #
        # LR
        # │\
        # │ \
        # │  \
        # │    ───
        # └────────── step
        #
        # 训练后期使用更小的学习率，
        # 有助于稳定收敛。
        lr_scheduler_type="cosine",


        # Warmup 占总训练 step 的 10%。
        #
        # 一开始不直接使用最大 learning rate，
        # 而是：
        #
        # 小 LR
        #  ↓
        # 逐渐增加
        #  ↓
        # 正常 LR
        #
        # 可以减少训练刚开始时参数突然发生剧烈变化。
        warmup_ratio=0.1,


        # =================================================
        # OPTIMIZATION
        # 优化相关设置
        # =================================================

        # Gradient Clipping
        #
        # 如果 gradient norm > 1.0，
        # 就对梯度进行缩放。
        #
        # 目的：
        # 防止 gradient explosion。
        max_grad_norm=1.0,


        # Adam optimizer 的 β1。
        #
        # 主要控制一阶动量：
        # 即过去 gradient 的指数移动平均。
        adam_beta1=0.9,


        # Adam optimizer 的 β2。
        #
        # 控制 gradient 平方的指数移动平均。
        #
        # 默认 Adam 常常是 0.999，
        # 这里使用 0.95，
        # 可以让二阶统计量对最近梯度变化响应更快。
        adam_beta2=0.95,


        # Weight Decay
        #
        # 对模型参数增加一定惩罚，
        # 防止权重无限增大，
        # 有一定正则化作用。
        #
        # 可以帮助降低 overfitting 风险。
        weight_decay=0.1,


        # =================================================
        # MONITORING
        # 日志 / 验证 / checkpoint
        # =================================================

        # 每多少个 training step 打印一次日志。
        #
        # 通常包括：
        # loss
        # reward
        # learning rate
        # gradient norm
        # 等。
        logging_steps=config.logging_steps,


        # 每多少 step 进行一次 evaluation。
        eval_steps=config.eval_steps,


        # 每多少 step 保存一次 checkpoint。
        save_steps=config.save_steps,


        # 最多保存多少个 checkpoint。
        #
        # 超过这个数量后，
        # 较旧 checkpoint 会被删除，
        # 避免占用太多硬盘空间。
        save_total_limit=config.save_total_limit,


        # =================================================
        # TECHNICAL SETTINGS
        # 技术配置
        # =================================================

        # 固定随机种子。
        #
        # 使随机初始化、sampling 等过程
        # 尽可能可复现（reproducibility）。
        seed=config.seed,


        # 如果 CUDA GPU 可用，
        # 则使用 bfloat16。
        #
        # BF16：
        # - 比 FP32 更省显存
        # - 通常比 FP16 数值范围更大
        # - 大模型训练中通常比较稳定
        bf16=True if torch.cuda.is_available() else False,


        # 不启用 FP16。
        #
        # 因为这里优先使用 BF16。
        fp16=False,


        # Trainer 默认可能会删除
        # dataset 中模型 forward() 没有直接使用的字段。
        #
        # GRPO 的 reward 计算过程中，
        # 可能还需要 answer / question / prompt 等字段，
        # 所以这里设置 False，保留这些字段。
        remove_unused_columns=False,


        # 不自动把训练结果上传到 Hugging Face Hub。
        push_to_hub=False,


        # 不把日志发送到 wandb、TensorBoard
        # 等外部 tracking 平台。
        report_to=[],
    )

    print("✅ GRPO configuration created")


    # =====================================================
    # 返回训练配置 + 正确答案查询表
    # =====================================================
    #
    # 注意这里还没有：
    #
    # trainer = GRPOTrainer(...)
    #
    # 所以严格来说，
    # 这里创建的是 GRPO configuration，
    # 而不是完整的 trainer。

    return GRPO_config, prompt2ans


# =========================================================
# 调用函数，创建 GRPO 配置
# =========================================================

GRPO_config, prompt2ans = create_grpo_trainer(
    config,
    model,
    tokenizer,
    train_dataset,
    eval_dataset,
    reward_model,
    test_dataset
)

print("\nConfiguration ready for training!")

In [ ]:
# ============================================
# Define the GRPO Trainer Creation Function
# ============================================

def create_grpo_trainer(config, model, tokenizer, train_dataset, eval_dataset, reward_model, test_dataset=None):
    """
    Create and configure the GRPO trainer.

    This function sets up everything needed for GRPO training:
    1. Configuration for the training process
    2. Reward computation function
    3. The trainer itself

    Args:
        config: Training configuration
        model: The language model to train
        tokenizer: Tokenizer for the model
        train_dataset: Training data
        eval_dataset: Validation data
        reward_model: Your reward scoring system
        test_dataset: Test data for evaluation

    Returns:
        A configured GRPO trainer ready to train
    """

    print("🔧 Creating GRPO trainer...\n")

    # Build a dictionary for fast answer lookup
    # This maps each prompt to its correct answer
    # O(1) lookup is much faster than searching through the dataset
    prompt2ans = {}
    for item in train_dataset:
        prompt2ans[item['prompt']] = (item['answer'], item['question'])
    print(f"Built answer lookup with {len(prompt2ans):,} entries")

    # Configure GRPO training parameters
    # In most cases, you don't need to modify the parameters below
    GRPO_config = GRPOConfig(
        # ===== GRPO SPECIFIC =====
        num_generations=config.num_generations,  # Generate 12 answers per question
        temperature=config.temperature,  # Randomness in generation
        generation_kwargs={
            "max_new_tokens": config.max_new_tokens,
            "temperature": config.temperature,
            "top_p": 0.95,  # Only consider top 95% probability tokens
            "do_sample": True,  # Enable random sampling
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        },
        generation_batch_size=min(96, config.per_device_train_batch_size * config.num_generations),

        # ===== OUTPUT SETTINGS =====
        output_dir=config.output_dir,

        # ===== TRAINING DURATION =====
        num_train_epochs=config.num_train_epochs,

        # ===== BATCH SETTINGS =====
        per_device_train_batch_size=config.per_device_train_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,

        # ===== LEARNING SETTINGS =====
        learning_rate=config.learning_rate,
        lr_scheduler_type="cosine",  # Gradually reduce learning rate
        warmup_ratio=0.1,  # Start with lower learning rate for stability

        # ===== OPTIMIZATION =====
        max_grad_norm=1.0,  # Clip gradients to prevent explosions
        adam_beta1=0.9,  # Momentum parameter
        adam_beta2=0.95,  # Better for RL than default 0.999
        weight_decay=0.1,  # L2 regularization to prevent overfitting

        # ===== MONITORING =====
        logging_steps=config.logging_steps,
        eval_steps=config.eval_steps,
        save_steps=config.save_steps,
        save_total_limit=config.save_total_limit,

        # ===== TECHNICAL SETTINGS =====
        seed=config.seed,
        bf16=True if torch.cuda.is_available() else False,  # Use bfloat16 on GPU
        fp16=False,  # Don't use float16 (less stable)
        remove_unused_columns=False,
        push_to_hub=False,  # Don't upload to Hugging Face
        report_to=[],  # Disable external logging
    )

    print("✅ GRPO configuration created")
    return GRPO_config, prompt2ans

# Create the configuration
GRPO_config, prompt2ans = create_grpo_trainer(config, model, tokenizer, train_dataset, eval_dataset, reward_model, test_dataset)
print("\nConfiguration ready for training!")

🔧 Creating GRPO trainer...

Built answer lookup with 5,978 entries
✅ GRPO configuration created

Configuration ready for training!


### Reward compuation function defined with detailed logging

This function will:
  1. Score each generated answer with detailed logs
  2. Show question, response, and reward for each generation
  3. Compare answers within groups
  4. Return relative rewards for training
  5. Log group statistics for debugging

In this function you can find the implementation of both the `mean_reward` and `normalized_rewards`, which you saw in the video.

enumerate() 是 Python 自带函数。它的作用是：

遍历一个列表的同时，既拿到“序号”，又拿到“元素本身”。

比如：

unique_prompts = ["题目A", "题目B", "题目C"]

for i, unique_prompt in enumerate(unique_prompts):
    print(i, unique_prompt)

输出：

0 题目A
1 题目B
2 题目C

一个 prompt
   ↓
生成多个 completion
   ↓
[y1, y2, y3, y4, ...]
   ↓
Reward Model 分别打分
   ↓
[r1, r2, r3, r4, ...]
   ↓
计算组内平均值
   ↓
mean_reward
   ↓
每个 reward - mean_reward
   ↓
normalized rewards
   ↓
好的答案 → 正数
差的答案 → 负数
   ↓
GRPO 用它更新模型

# ============================================
# 定义 Reward 计算函数
# ============================================

def compute_rewards(prompts: List[str], completions: List[str], **kwargs):
    """
    计算模型生成答案的 reward，并输出详细日志。

    这是 GRPO 中非常核心的一部分：
    模型针对同一个 prompt 会生成多个答案，
    然后我们给每个答案打分，
    再在同一组答案内部进行相对比较。

    整体流程：
    1. 根据 prompt 对 completions 进行分组
    2. 使用 reward_model 给每个 completion 打分
    3. 在同一个 prompt 的生成结果内部做 reward 归一化
    4. 按照原始顺序返回 normalized rewards

    Args:
        prompts:
            prompt 列表。
            因为一个 prompt 会生成多个 completion，
            所以同一个 prompt 通常会重复出现多次。

            例如：
            [
                "2+2=?", "2+2=?", "2+2=?",
                "3+3=?", "3+3=?", "3+3=?"
            ]

        completions:
            模型生成的回答列表。

            例如：
            [
                "4",
                "The answer is 4",
                "5",
                "6",
                "The answer is 6",
                "7"
            ]

        **kwargs:
            接收额外参数。
            即使这里暂时不用，也可以避免 Trainer
            传入额外参数时报错。

    Returns:
        rewards:
            与 completions 一一对应的 normalized reward 列表。
    """

    # 最终返回的 reward 列表
    #
    # rewards[i] 对应 completions[i]
    rewards = []


    # =====================================================
    # 1. 输出当前 batch 的基本信息
    # =====================================================

    # 打印一条分隔线，方便查看日志
    logger.info(f"\n{'='*80}")

    logger.info(f"GRPO REWARD COMPUTATION")

    # 当前 batch 总共有多少个模型生成结果
    logger.info(f"Total completions: {len(completions)}")

    # prompt 数量
    #
    # 注意：
    # 这里包含重复 prompt。
    #
    # 例如：
    # 一个 prompt 生成 12 次，
    # 那这个 prompt 会出现 12 次。
    logger.info(f"Total prompts: {len(prompts)}")

    # config.num_generations：
    # 每个 prompt 理论上应该生成多少个 completion。
    #
    # 例如：
    #
    # config.num_generations = 12
    #
    # 每个数学题应该生成 12 个不同答案。
    logger.info(
        f"Expected generations per prompt: {config.num_generations}"
    )

    logger.info(f"{'='*80}")


    # =====================================================
    # 2. 找出所有“不重复”的 prompt
    # =====================================================

    # seen：
    # 用于记录已经出现过的 prompt。
    #
    # set 的查找速度平均为 O(1)。
    seen = set()

    # 保存去重后的 prompt，
    # 同时保持它们原来的出现顺序。
    unique_prompts = []

    for p in prompts:

        # 如果这个 prompt 之前没有出现过
        if p not in seen:

            # 加入 unique_prompts
            unique_prompts.append(p)

            # 标记为已经出现过
            seen.add(p)


    # 当前 batch 里真正有多少道不同的问题
    num_unique_prompts = len(unique_prompts)

    logger.info(
        f"Calculated unique prompts: {num_unique_prompts}"
    )


    # =====================================================
    # 3. Debug：打印前几个模型生成结果
    # =====================================================

    # min(4, len(completions))
    #
    # 表示：
    # 最多打印前 4 个 completion。
    #
    # 如果 completion 总数不到 4 个，
    # 就打印实际存在的数量。
    for i in range(min(4, len(completions))):

        # completions[i][:80]
        #
        # 只显示前 80 个字符，
        # 防止日志太长。
        logger.info(
            f"Completion {i+1}: {completions[i][:80]}..."
        )


    # =====================================================
    # 4. 对每一个 unique prompt 单独处理
    # =====================================================

    for i, unique_prompt in enumerate(unique_prompts):

        # -------------------------------------------------
        # 4.1 找到这个 prompt 在原始 prompts 中的位置
        # -------------------------------------------------

        # enumerate(prompts) 会产生：
        #
        # (0, prompt_0)
        # (1, prompt_1)
        # (2, prompt_2)
        # ...
        #
        # 如果当前 p == unique_prompt，
        # 就记录对应的 idx。
        #
        # 例如：
        #
        # prompts =
        # [A, A, A, B, B, B]
        #
        # 当前 unique_prompt = A
        #
        # prompt_indices =
        # [0, 1, 2]
        prompt_indices = [
            idx
            for idx, p in enumerate(prompts)
            if p == unique_prompt
        ]


        # -------------------------------------------------
        # 4.2 获取这个 prompt 对应的所有 completions
        # -------------------------------------------------

        # 根据刚才找到的 index，
        # 从 completions 中取出对应生成结果。
        #
        # 例如：
        #
        # prompt_indices = [0,1,2]
        #
        # 那么取：
        #
        # completions[0]
        # completions[1]
        # completions[2]
        group_completions = [
            completions[idx]
            for idx in prompt_indices
        ]


        # -------------------------------------------------
        # 4.3 获取正确答案
        # -------------------------------------------------

        # prompt2ans 是前面建立的 dictionary：
        #
        # prompt
        #   ↓
        # (correct_answer, question)
        #
        # 例如：
        #
        # prompt2ans["Solve 2+2"]
        # =
        # ("4", "What is 2+2?")
        #
        # .get() 的第二个参数：
        #
        # (0.0, "Unknown question")
        #
        # 是默认值。
        #
        # 如果当前 prompt 没有在 dictionary 中找到，
        # 就不会直接报 KeyError，
        # 而是使用这个默认值。
        correct_answer, question = prompt2ans.get(
            unique_prompt,
            (0.0, "Unknown question")
        )


        # =================================================
        # 输出当前这一组 prompt 的信息
        # =================================================

        logger.info(
            f"\n--- Unique Prompt {i+1}/{num_unique_prompts} ---"
        )

        # 只打印 question 前 150 个字符
        logger.info(
            f"Question: {question[:150]}..."
        )

        # 标准答案
        logger.info(
            f"Expected Answer: {correct_answer}"
        )

        # 当前 prompt 实际生成了多少个答案
        logger.info(
            f"Number of generations for this prompt: "
            f"{len(group_completions)}"
        )


        # =================================================
        # 5. 给这一组每一个 completion 计算 reward
        # =================================================

        group_rewards = []

        for j, completion in enumerate(group_completions):

            logger.info(
                f"\n  Generation "
                f"{j+1}/{len(group_completions)}:"
            )


            # ---------------------------------------------
            # 核心：Reward Model 对答案进行打分
            # ---------------------------------------------
            #
            # reward_model.compute_reward(...)
            #
            # 输入：
            #
            # completion：
            #   模型生成的答案
            #
            # correct_answer：
            #   标准答案
            #
            # question：
            #   原始问题
            #
            # 输出：
            #
            # reward：
            #   一个数值分数
            #
            # 比如：
            #
            # 正确答案 → 1.0
            # 部分正确 → 0.5
            # 错误答案 → 0.0
            #
            # 具体怎么算，
            # 取决于你定义的 reward_model。
            reward = reward_model.compute_reward(
                completion,
                correct_answer,
                question
            )


            # 保存当前 completion 的原始 reward
            group_rewards.append(reward)


        # =================================================
        # 6. 在组内进行 Reward Normalize
        # =================================================
        #
        # GRPO 的核心思想之一：
        #
        # 不只是看一个答案自己的绝对 reward，
        # 而是看：
        #
        # “这个答案相对于同一道题其他答案表现如何？”
        #
        # 假设：
        #
        # 原始 rewards：
        #
        # [1.0, 0.8, 0.2, 0.0]
        #
        # mean =
        #
        # (1 + 0.8 + 0.2 + 0) / 4
        # = 0.5
        #
        # mean-center 后：
        #
        # [
        #   1.0 - 0.5 =  0.5,
        #   0.8 - 0.5 =  0.3,
        #   0.2 - 0.5 = -0.3,
        #   0.0 - 0.5 = -0.5
        # ]
        #
        # 正数：
        #   比同组平均表现更好
        #
        # 负数：
        #   比同组平均表现更差

        if len(group_rewards) > 1:

            # 当前这一组答案的平均 reward
            mean_reward = (
                sum(group_rewards)
                / len(group_rewards)
            )


            # ------------------------------------------------
            # Mean-centering
            # ------------------------------------------------
            #
            # 每一个 reward 减去组内平均值。
            #
            # 注意：
            # 这里没有除以 standard deviation，
            # 所以严格来说只是 mean-centering，
            # 不是完整的 z-score normalization。
            normalized_rewards = [
                r - mean_reward
                for r in group_rewards
            ]

        else:

            # 如果只有一个 completion：
            #
            # 没有其他答案可以进行“相对比较”。
            #
            # 所以设置 reward = 0。
            #
            # 因为：
            #
            # r - mean(r)
            # =
            # r - r
            # =
            # 0
            normalized_rewards = [0.0]


        # =================================================
        # 7. 把 normalized rewards 放回原来的顺序
        # =================================================
        #
        # 因为前面我们按照 prompt 分组进行了计算，
        # 现在要重新恢复：
        #
        # completions[i]
        # ↕
        # rewards[i]
        #
        # 的对应关系。

        for idx, norm_reward in zip(
            prompt_indices,
            normalized_rewards
        ):

            # 如果 rewards 当前长度不够，
            # 先补 0。
            #
            # 例如：
            #
            # idx = 5
            #
            # 但 rewards 当前长度只有 3，
            # 就需要先扩展到 index 5。
            if len(rewards) <= idx:

                rewards.extend(
                    [0] * (
                        idx - len(rewards) + 1
                    )
                )


            # 把 normalized reward
            # 放到原 completion 对应的位置。
            rewards[idx] = norm_reward


        # =================================================
        # 8. 统计这一组的 reward 信息
        # =================================================

        # 原始 reward 平均值
        avg_reward = (
            sum(group_rewards)
            / len(group_rewards)
        )

        # 当前组最高 reward
        max_reward = max(group_rewards)

        # 当前组最低 reward
        min_reward = min(group_rewards)

        # normalized reward 平均值
        #
        # 因为做了 mean-centering，
        # 理论上这个值应该非常接近 0。
        avg_norm = (
            sum(normalized_rewards)
            / len(normalized_rewards)
        )


        # 输出原始 reward 统计
        logger.info(
            f"\n  Group Summary: "
            f"Avg={avg_reward:.3f}, "
            f"Max={max_reward:.3f}, "
            f"Min={min_reward:.3f}"
        )


        # 输出归一化后的 reward
        logger.info(
            f"  Normalized: "
            f"Avg={avg_norm:.3f}, "
            f"Rewards="
            f"{[f'{r:.3f}' for r in normalized_rewards]}"
        )


    # =====================================================
    # 9. 整个 batch 的统计
    # =====================================================

    # rewards 非空：
    #     计算整个 batch normalized reward 的平均值
    #
    # rewards 为空：
    #     设置为 0，防止除以 0
    overall_avg = (
        sum(rewards) / len(rewards)
        if rewards
        else 0
    )


    logger.info(f"\n{'='*80}")

    logger.info(
        f"BATCH SUMMARY: "
        f"Overall Average Reward = {overall_avg:.3f}"
    )

    logger.info(f"{'='*80}\n")


    # =====================================================
    # 10. 返回 reward
    # =====================================================
    #
    # Trainer 后续会使用这些 reward
    # 来计算 GRPO 的优化目标并更新模型参数。
    return rewards

In [ ]:
# ============================================
# Define the Reward Computation Function
# ============================================

def compute_rewards(prompts: List[str], completions: List[str], **kwargs):
    """
    Compute rewards for generated completions with detailed logging.

    This is the heart of GRPO - it scores each generated answer
    and normalizes within groups for relative comparison.

    Process:
    1. Group completions by their prompt
    2. Score each completion with detailed logging
    3. Normalize scores within each group
    4. Return normalized rewards

    Args:
        prompts: List of prompts (with duplicates for each generation)
        completions: List of generated responses

    Returns:
        List of normalized rewards
    """

    rewards = []

    # Log batch information
    logger.info(f"\n{'='*80}")
    logger.info(f"GRPO REWARD COMPUTATION")
    logger.info(f"Total completions: {len(completions)}")
    logger.info(f"Total prompts: {len(prompts)}")
    logger.info(f"Expected generations per prompt: {config.num_generations}")
    logger.info(f"{'='*80}")

    # Get unique prompts (remove duplicates)
    seen = set()
    unique_prompts = []
    for p in prompts:
        if p not in seen:
            unique_prompts.append(p)
            seen.add(p)

    num_unique_prompts = len(unique_prompts)
    logger.info(f"Calculated unique prompts: {num_unique_prompts}")

    # Debug: Print samples
    for i in range(min(4, len(completions))):
        logger.info(f"Completion {i+1}: {completions[i][:80]}...")

    # Process each unique prompt and its completions
    for i, unique_prompt in enumerate(unique_prompts):
        # Find all completions for this prompt
        prompt_indices = [idx for idx, p in enumerate(prompts) if p == unique_prompt]
        group_completions = [completions[idx] for idx in prompt_indices]

        # Get correct answer from your lookup dictionary
        correct_answer, question = prompt2ans.get(unique_prompt, (0.0, "Unknown question"))

        logger.info(f"\n--- Unique Prompt {i+1}/{num_unique_prompts} ---")
        logger.info(f"Question: {question[:150]}...")
        logger.info(f"Expected Answer: {correct_answer}")
        logger.info(f"Number of generations for this prompt: {len(group_completions)}")

        # Compute reward for each completion with detailed logging
        group_rewards = []
        for j, completion in enumerate(group_completions):
            logger.info(f"\n  Generation {j+1}/{len(group_completions)}:")
            reward = reward_model.compute_reward(completion, correct_answer, question)
            group_rewards.append(reward)

        # Normalize rewards within the group (mean-centering)
        # This makes rewards relative: positive = better than average, negative = worse
        if len(group_rewards) > 1:
            mean_reward = sum(group_rewards) / len(group_rewards)
            # Mean-center only - no standard deviation scaling
            normalized_rewards = [r - mean_reward for r in group_rewards]
        else:
            # Single generation - no comparison possible
            normalized_rewards = [0.0]

        # Add normalized rewards in correct order
        for idx, norm_reward in zip(prompt_indices, normalized_rewards):
            if len(rewards) <= idx:
                rewards.extend([0] * (idx - len(rewards) + 1))
            rewards[idx] = norm_reward

        # Log group statistics matching your format
        avg_reward = sum(group_rewards) / len(group_rewards)
        max_reward = max(group_rewards)
        min_reward = min(group_rewards)
        avg_norm = sum(normalized_rewards) / len(normalized_rewards)
        logger.info(f"\n  Group Summary: Avg={avg_reward:.3f}, Max={max_reward:.3f}, Min={min_reward:.3f}")
        logger.info(f"  Normalized: Avg={avg_norm:.3f}, Rewards={[f'{r:.3f}' for r in normalized_rewards]}")

    overall_avg = sum(rewards) / len(rewards) if rewards else 0
    logger.info(f"\n{'='*80}")
    logger.info(f"BATCH SUMMARY: Overall Average Reward = {overall_avg:.3f}")
    logger.info(f"{'='*80}\n")

    return rewards

In [ ]:
# ============================================
# Create the GRPO Trainer
# ============================================

print("Creating the GRPO trainer...\n")

# Initialize the GRPO trainer with all the components
trainer = GRPOTrainer(
    model=model,  # The language model to train
    reward_funcs=compute_rewards,  # Function to compute rewards
    args=GRPO_config,  # Training configuration
    train_dataset=train_dataset,  # Training data
    eval_dataset=eval_dataset,  # Validation data
    processing_class=tokenizer,  # Tokenizer
)

# Add the evaluation callback
trainer.add_callback(eval_callback)
print("✅ Added evaluation callback for progress tracking")

print("\n✅ GRPO Trainer created successfully!")
print("\nTraining Summary:")
print(f"  Model: DeepSeek Math 7B")
print(f"  Dataset: GSM8K math problems")
print(f"  Training examples: {len(train_dataset):,}")
print(f"  Generations per prompt: {config.num_generations}")
print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
print(f"  Total training steps: ~{len(train_dataset) // (config.per_device_train_batch_size * config.gradient_accumulation_steps) * config.num_train_epochs}")
print(f"\nThe model will now learn to solve math problems better!")

Creating the GRPO trainer...

✅ Added evaluation callback for progress tracking

✅ GRPO Trainer created successfully!

Training Summary:
  Model: DeepSeek Math 7B
  Dataset: GSM8K math problems
  Training examples: 5,978
  Generations per prompt: 12
  Effective batch size: 64
  Total training steps: ~465

The model will now learn to solve math problems better!


## Train the Model with GRPO! (Ungraded Part) <a id="trainmodelwithgrpo"></a>

### What Happens During Training

During each training step:
1. **Select** a batch of math problems
2. **Generate** 12 different solutions for each
3. **Score** each solution with the reward model
4. **Compare** solutions within each group
5. **Update** the model to prefer better solutions

### Training Time (With GPU)

- To finish the full training, it will take dozens of hours. (Generally, it's not necessary to finish the full training schedule.)
- With proper reward functions, you can notice that the Eval Acc is showing an upward trend within 1 or 2 hours of training
- With proper reward functions, the Eval Acc could go above 40% within 10 to 15 hours of training.

### What to Watch

- **Loss**: Should decrease (model is learning)
- **Accuracy**: Should increase (getting more problems right)
- **Rewards**: Should increase (generating better solutions)

### Important Notes

- Training will save checkpoints periodically
- You can resume if training is interrupted
- Evaluation results every 20 steps
- Please analyze the logs under the grpo_logs folder to imporve the reward function

In [ ]:
# ============================================
# Start GRPO Training!
# ============================================

print("You have successfully built the trainer. Next, you can use it to train GRPO. However, since the training process takes over 10 hours, it will not be conducted in this lab. You can run it on other GPU resources.")


You have successfully built the trainer. Next, you can use it to train GRPO. However, since the training process takes over 10 hours, it will not be conducted in this lab. You can run it on other GPU resources.


## Summary <a id="summary"></a>

### Congratulations!

You've successfully completed the GRPO training lab! You've learned how to use reinforcement learning to improve a language model's ability to solve math problems.

### What You've Learned

1. **GRPO Fundamentals**
   - How to generate multiple responses per prompt
   - Why relative comparison works better than absolute rewards
   - How group normalization stabilizes training

2. **Practical Implementation**
   - Setting up a reward model for math problems
   - Configuring GRPO training parameters
   - Monitoring training progress
   - Evaluating model improvements

3. **Key Insights**
   - More generations per prompt = better comparison
   - Partial credit rewards help learning
   - Evaluation on held-out data is crucial

### Next Steps

To further improve your results:

2. **Adjust Temperature**: Experiment with values between 0.5-1.0
3. **Increase Generations**: Try 16 or 20 generations per prompt
4. **Fine-tune Rewards**: Adjust the partial credit system
5. **Try Other Datasets**: Apply GRPO to different tasks


### Additional Resources

- **TRL Documentation**: [https://huggingface.co/docs/trl](https://huggingface.co/docs/trl)
- **GRPO Paper**: [arXiv:2402.03300](https://arxiv.org/abs/2402.03300)
- **GSM8K Dataset**: [OpenAI GitHub](https://github.com/openai/grade-school-math)
- **DeepSeek Models**: [Hugging Face](https://huggingface.co/deepseek-ai)

### Achievement Unlocked!

You've successfully:
- ✅ Implemented GRPO from scratch
- ✅ Trained a 7B parameter model
- ✅ Improved math problem-solving accuracy
- ✅ Learned modern RL techniques for LLMs

### Final Thoughts

GRPO is just one of many post-training techniques. The same principles you've learned here apply to:
- **PPO** (Proximal Policy Optimization)
- **DPO** (Direct Preference Optimization)
- **RLHF** (Reinforcement Learning from Human Feedback)

The future of AI involves not just pre-training large models, but also fine-tuning them for specific tasks using techniques like GRPO. You're now equipped with the knowledge to be part of that future!

### Thank You!

Thank you for completing this lab. We hope you found it educational and enjoyable. Keep experimenting, keep learning, and keep pushing the boundaries of what's possible with AI!
